# 

In [ ]:
import pandas as pd 
from scipy.spatial import distance
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from multiprocessing import Pool
import tqdm
from itertools import combinations
from scipy import stats
import seaborn as sns; sns.set_theme(style='white')
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator
from Levenshtein import distance as lev
from matplotlib_venn import venn2
import networkx as nx
Bio = pd.read_csv('/Users/aelazar/Downloads/BIOGRID-ALL-4.4.215.tab3.txt',delimiter='\t',header=0)
HPA_frac_df = pd.read_pickle('HPA_frac_df.pkl')

Bio_human = Bio[Bio['Organism ID Interactor A']==9606]
Bio_human = Bio_human.rename(columns={"SWISS-PROT Accessions Interactor A": "acc1", "SWISS-PROT Accessions Interactor B": "acc2"})
Bio_human = Bio_human.drop(index=Bio_human[(Bio_human.acc2=='-')|(Bio_human.acc1=='-')].index)
Bio_human = Bio_human.reset_index(drop=True)
group = Bio_human[['acc1', 'acc2']].agg(frozenset, axis=1)

Bio_human = (Bio_human
 .groupby(group, as_index=False)  # you can also group by [group, 'Score']
 .agg(**{c: (c, 'first') for c in Bio_human},
      Duplicates=('#BioGRID Interaction ID', 'count'),
     )
)
df = Bio_human
ind = set.union(set(df[df.acc1.str.find('|')!=-1].index),
set(df[df.acc2.str.find('|')!=-1].index))
Bio_human = Bio_human.drop(index=ind)
df  = Bio_human
#TMs = set(topology_df.acc)
dSAMs =pd.read_pickle('../files/dSAM_MPs_v_paper_v2_new_locs.pkl')
AF2= pd.read_pickle('../files/AF2_with_ENSG.pkl')
from adjustText import adjust_text
hTMs = pd.read_pickle('../files/hTMs_uniprot_v1.pkl')
acc2name = pd.read_csv('/Users/aelazar/notebooks_for_paper/BioGrid_anaysis/uniprot-download_true_fields_accession_2Creviewed_2Cid_2Cprotein_nam-2023.03.06-18.50.31.99.tsv',delimiter='\t')
sub_TMs = list(set(hTMs.acc))#list(set(HPA_frac_df[HPA_frac_df.gmm_new==0].acc))
non_TMs = set.difference(set.union(set(df.acc1),set(df.acc2)),set(hTMs.acc))

In [ ]:
s1 = set(df[(df['Organism ID Interactor A']==9606)&(df['Organism ID Interactor B']==9606)].acc1)
s2 = set(df[(df['Organism ID Interactor A']==9606)&(df['Organism ID Interactor B']==9606)].acc2)
len(set.union(s1,s2))


In [ ]:
len(df[(df['Organism ID Interactor A']==9606)&(df['Organism ID Interactor B']==9606)])

In [ ]:
uniport_annotation  = pd.read_csv('/Users/aelazar/Downloads/idmapping_2023_11_15.tsv',delimiter='\t')
idr_acc = []
for j in sub_TMs:
    if AF2[AF2.acc==j].pLDDT.values[0]!='A':
        idr_acc.append({'acc':j,'idr':len([i for i in AF2[AF2.acc==j].pLDDT.values[0] if i<70])})
idr_df = pd.DataFrame(idr_acc)

In [ ]:
non_TMs = set.difference(set.union(set(df.acc1),set(df.acc2)),set(hTMs.acc))
mdSAMs = set(dSAMs.acc)
ER = set(HPA_frac_df[HPA_frac_df.gmm_new==1].index.values)
PM = set(HPA_frac_df[HPA_frac_df.gmm_new==0].index.values)
Golgi = set(HPA_frac_df[HPA_frac_df.gmm_new==3].index.values)
Endo_Lyso = set(HPA_frac_df[HPA_frac_df.gmm_new==2].index.values)
Other = set(HPA_frac_df[HPA_frac_df.gmm_new==4].index.values)

In [ ]:
def Graph_organelle(organelle=PM,mdSAMs=mdSAMs,min_egde = 1, max_edge=90,non_TMs=non_TMs):
    sub_TMs = list(organelle)
    nodes_for_graph = set.union(non_TMs,set(sub_TMs))
    mdSAM_PM  = set.intersection(organelle,mdSAMs)#(set.intersection(set(HPA_frac_df[HPA_frac_df.gmm_new==0].acc),set(dSAMs.acc)))
    non_mdSAM_PM  = set.difference(set(sub_TMs),mdSAM_PM)

    G = nx.Graph()
    l = []
    #accs = (set.union(set(df.acc1),set(df.acc2))) ## Nodes 
    for acc in nodes_for_graph:
        if acc in list(mdSAM_PM):
            l.append((acc, {"mdSAM": "yes"}))
        if acc in list(non_mdSAM_PM):
            l.append((acc, {"mdSAM": "no"}))
    G.add_nodes_from(l)
    for ind in tqdm.tqdm(df.index):
        if ((df.loc[ind,'acc1'] in nodes_for_graph) and (df.loc[ind,'acc2'] in nodes_for_graph)):
            if ((df.loc[ind,'acc1'] in sub_TMs) and (df.loc[ind,'acc2'] in sub_TMs)):
                continue#print(df.loc[ind,'acc1'],df.loc[ind,'acc2'])
            if ((df.loc[ind,'acc1']  not in sub_TMs) and (df.loc[ind,'acc2'] not in sub_TMs)):
                continue#print(df.loc[ind,'acc1'],df.loc[ind,'acc2'])
            else:
                G.add_edge(df.loc[ind,'acc1'], df.loc[ind,'acc2'])
    z = []
    for i in G.nodes:
        if len(G[i])<=min_egde:
            z.append(i)
        if len(G[i])>=max_edge:
            z.append(i)
            #G.remove_node(i)
    G.remove_nodes_from(z)
    # zz=[]
    # for i in G.nodes:
    #     if len(G[i])<=1:
    #         zz.append(i)
    # G.remove_nodes_from(zz)
    return G

In [ ]:
G_PM = Graph_organelle(organelle=PM)
G_ER = Graph_organelle(organelle=ER)
G_Golgi = Graph_organelle(organelle=Golgi)
G_Endo_Lyso = Graph_organelle(organelle=Endo_Lyso)
G_Other = Graph_organelle(organelle=Other)

In [ ]:
mdSAM_PM  = set.intersection(set(G_PM.nodes),mdSAMs)
mdSAM_ER  =  set.intersection(set(G_ER.nodes),mdSAMs)
mdSAM_Golgi  =  set.intersection(set(G_Golgi.nodes),mdSAMs)
mdSAM_Endo_Lyso =  set.intersection(set(G_Endo_Lyso.nodes),mdSAMs)
mdSAM_Other =  set.intersection(set(G_Other.nodes),mdSAMs)

In [ ]:
def G_hypergeom(G,acc,mdSAMs=mdSAM_PM):
    total_runners =  len(set.intersection(set(list(G.nodes)),set(sub_TMs))) # No. of hTMs Nodes in the Graph 
    number_team_A = len(set.intersection(set(list(G.nodes)),mdSAMs)) # number mdSAM nodes in the Graph  
    number_winners = len(list(G[acc]))## No. of PPIs i.e Edges 
    number_win_from_A = len(set.intersection(set(list(G[acc])),mdSAMs)) ## PPIs that are mdSAM
    #print(total_runners,number_team_A,number_winners,number_win_from_A)
    expected = (number_team_A*number_winners)/total_runners
    p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
    #print(p)
    if  number_win_from_A-expected >0:
        p = 1-p 
    return number_win_from_A,expected,p 

In [ ]:
mdSAM_PM  = set.intersection(set(G_PM.nodes),mdSAMs)
G_hypergeom(G_PM,'P63000',mdSAM_PM)

In [ ]:
def G2df(G,mdSAMs = mdSAMs):
    G_hypergeom_list = []
    for i in tqdm.tqdm(set.intersection(set(list(G.nodes)),non_TMs)):
        n,e,p  = G_hypergeom(G,i,mdSAMs)
        if acc2name[acc2name.From==i]['Gene Names'].empty==False:
            gene = acc2name[acc2name.From==i]['Gene Names'].values[0].split(' ')[0]
        else:
            gene= np.NaN
        G_hypergeom_list.append({'acc':i,'no_edges':n,'expected':e,'p':p,'no_edges_total':len(set.intersection(set(list(G[i])),set(sub_TMs))),
                                 'gene':gene,'round':'real'})
    G_hypergeom_df  = pd.DataFrame(G_hypergeom_list)
    G_hypergeom_df = G_hypergeom_df[G_hypergeom_df.no_edges_total>=1]
    G_hypergeom_df['obs_exp'] = (G_hypergeom_df['no_edges']-G_hypergeom_df['expected'])#/G_hypergeom_df['expected']
    G_hypergeom_df['fc'] = G_hypergeom_df['no_edges']/ G_hypergeom_df['expected']
    G_hypergeom_df['log_p'] = -1*np.log10(G_hypergeom_df['p'])
    G_hypergeom_df['fdr'] = fdr(G_hypergeom_df.p)
    G_hypergeom_df['log_fdr'] = -1*np.log10(G_hypergeom_df['fdr'])
    
    G_plot = G_hypergeom_df#G_hypergeom_df[(G_hypergeom_df.p!=0)&(G_hypergeom_df.no_edges_total<320)]
    G_plot  = G_plot[~pd.isna(G_plot.gene)]
    return G_plot

In [ ]:
def fdr(p_vals):

    from scipy.stats import rankdata
    ranked_p_values = rankdata(p_vals)
    fdr = p_vals * len(p_vals) / ranked_p_values
    fdr[fdr > 1] = 1

    return fdr

In [ ]:
#set.intersection(set(list(G_PM.nodes)),non_TMs)
G_plot_PM = G2df(G_PM,mdSAMs=mdSAM_PM)#pd.Series(list(PM)).sample(71)
G_plot_ER = G2df(G_ER,mdSAMs=mdSAM_ER)
G_plot_Golgi = G2df(G_Golgi,mdSAMs=mdSAM_Golgi)
G_plot_Endo_Lyso = G2df(G_Endo_Lyso,mdSAMs=mdSAM_Endo_Lyso)
G_plot_Other = G2df(G_Other,mdSAMs=mdSAM_Other)


In [ ]:
E3_ligases = ['Q9UHB7','Q9UKV5','Q9NYG5','Q9P2G1','O15033','Q9Y4X5','O95376','Q99728','Q9NZS9','Q13490','Q13489','Q96CA5','Q96P09','P35226','Q7Z569','P38398','P22681','Q13191','Q9ULV8','Q75N03','Q8IYA8','Q9NPC3','Q99675','Q96EP1','O95628','Q8IWT3','Q6ZMK1','Q5T197','Q86Y01','Q86UW9','Q8N9I9','Q8TDB6','Q9Y2E6','Q86Y13','Q66K89','Q9NW38','Q7L622','Q8IYU2','Q9ULT8','Q5U5R9','Q5T447','Q9Y4D8','Q76N89','Q9P2P5','Q15751','O95714','Q15034','Q5GLZ8','Q9UII4','Q8IVU3','Q14527','Q7Z6Z7','Q8IU81','Q7Z5L9','Q9H1B7','Q96J02','Q9P0J7','Q8NEZ4','O14686','Q8TBB1','Q8N448','Q17RB8','Q1L5Z9','Q496Y0','Q6UWE0','O94822','Q7L5Y9','Q13233','Q8TCQ1','Q8NA82','A6NNE9','Q9P0N8','Q86UD3','Q9P2E8','Q9NX47','O60337','Q9H992','Q5T0T0','Q86YJ5','Q00987','O15151','Q03112','A1L020','Q6ZN04','Q5U5Q3','Q86XN8','O60291','Q86YT6','Q96AX9','O15344','Q9UJV3','Q9UHC7','Q9H000','Q13064','Q13434','P51948','Q9HC17','Q969V5','O75592','Q8WY64','P46934','Q96PU5','O76050','A8MQ27','Q96EH8','Q12986','Q6ZNB6','Q6VVB1','Q9Y314','Q8WV22','O60260','Q9BSM1','P35227','Q3KNV8','Q86SE9','Q9BYE7','Q9UPQ7','Q6ZMN7','Q96FA3','Q9HAT8','Q8N2H9','O60683','O00623','P28328','Q9BWX1','Q9P1Y6','Q8NG27','O43164','Q6DJT9','Q9UM63','P29590','Q13356','Q9UMS4','Q9NS91','P15918','Q13702','Q7Z6E9','Q9BYM8','P62877','Q5TC82','Q9HBD1','Q96PM5','Q8WZ73','O75677','O75678','O75679','A6NLU0','F8VTS6','Q6ZWI9','Q8NHY2','Q6PCD5','Q06587','Q13129','Q9NVW2','Q9H871','Q96G75','Q8N5U6','O00237','Q9Y3C5','Q6ZNA4','Q9ULX5','O15541','Q8IZP6','Q9Y508','Q9Y4L5','Q9H920','Q9H9V4','Q5XPI4','Q96EQ8','Q9BV68','Q8TEB7','O34567','Q86XS8','Q8WVZ7','Q8IUD6','Q8WVD3','Q8WU17','Q9UBS8','Q8WVD5','P50876','Q7Z419','Q96MT1','Q9NTX7','Q8N7C7','Q8NC42','Q9ULK6','Q2KHN1','Q8N8N0','Q96PX1','Q6ZSG1','Q96A37','Q9H6Y7','Q8IYW5','Q8NCN4','Q9BXT8','Q96K19','Q8N4F7','Q86T96','Q9P0P0','Q8N6D2','Q96D59','Q96GF1','Q9NXI6','Q5TA31','Q9NV58','Q6ZMZ0','Q99496','Q5VTR2','Q6ZRF8','Q9H0X6','Q495C1','A8MTL3','Q63HN8','Q8ND24','Q9Y6U7','Q9NWF9','Q8TC41','Q5W0B1','Q5VTB9','A6NCQ9','E7ERA6','P0DH78','M0QZC1','Q9Y225','Q96BH1','Q9BY78','Q96EP0','Q9H0A6','Q969K3','Q9H0F5','Q9H2S5','P78317','O75150','Q9H4P4','Q68DV7','Q7L0R7','Q99942','Q9Y252','Q9UBF6','O76064','Q5M7Z0','Q96EX2','Q96DX4','Q99590','Q7Z6J0','Q8TEC5','Q8TEJ3','Q149N8','Q8IUQ4','O43255','Q8IW03','Q9HCE7','Q9HAU4','Q9UNE7','Q86TM6','A0AVI4','Q9NS56','Q12933','Q13114','Q9BUZ4','O00463','Q9Y4K3','Q6Q0C0','Q9BWF2','Q9UDY6','Q96F44','O60858','Q9C019','Q9Y577','Q9C040','P19474','Q8IYM9','P36406','O15164','Q14258','Q12899','P14373','Q123263','O75382','Q9BZY9','Q13049','Q9UPN9','Q9BYJ4','Q9UPQ4','Q9NQ86','O94972','O00635','Q9HCM9','Q9C037','Q6P9F5','Q8WV44','Q8IWZ5','Q96BQ3','A6NCK2','Q9H8W5','Q7Z4K8','Q96LD4','Q8IWZ4','P0CI25','A6NDI0','P0CI26','C9J1S8','Q9C035','Q86XT4','Q9BSJ1','Q96A61','Q9BYV2','Q9BYV6','Q9BRZ2','Q8NG06','Q8IWR1','Q9C030','Q495X7','Q5EBN2','Q9BVG3','Q969Q1','A6NGJ6','A6NI03','A6NLI5','Q6PJ69','Q6ZTA4','Q6AZZ1','Q86WT6','Q9C029','Q2Q1W2','Q6ZMU5','Q86UV7','Q86UV6','A6NK02','I1YAP6','Q9BZR9','Q9C026','Q8N9V2','Q8N7C3','Q14669','P53804','Q05086','Q7Z3V4','Q15386','Q7Z6J8','Q14139','O95155','O94941','Q8IWV7','Q8IWV8','Q6ZT12','Q5T4S7','O95071','Q8N806','Q96T88','Q96PU4','Q9C0B0','Q9H9P5','Q9H270','Q9P253','P49754','Q8N3P4','Q6PJI9','Q8N9V3','Q9H0M0','O00308','P98170','Q9Y330','Q96JP5','O95159','P59817','Q9BYN7','Q8NB15','Q96K83','Q86UK7','Q8N7E2','Q8ND25','Q8NHG8','Q9ULT6','Q8WWF5','Q8NEG5','Q2QGD7']

In [ ]:
print('%d mdSAM nodes'%len(set.intersection(set(G_ER.nodes),mdSAM_ER)))
print('%d non-mdSAM nodes'%len(set.intersection(set(G_ER.nodes),set.difference(ER,mdSAM_ER))))
print('%d non-TM nodes'%len(set.intersection(set(G_PM.nodes),non_TMs)))


In [ ]:
print('%d mdSAM nodes'%len(set.intersection(set(G_PM.nodes),mdSAM_PM)))
print('%d non-mdSAM nodes'%len(set.intersection(set(G_PM.nodes),set.difference(PM,mdSAM_PM))))
print('%d non-TM nodes'%len(set.intersection(set(G_PM.nodes),non_TMs)))


In [ ]:
import glob 
d =[]
for acc in tqdm.tqdm(glob.glob('/Users/aelazar/notebooks_for_paper/interpro/G_plot/*')):
    id_  = acc.split('/')[-1].split('.')[0]
    if id_ !='accs':
        x  = pd.read_json(acc)
        for i in range(len(x)):
            if pd.DataFrame([x.iloc[i].results['metadata']])['type'].values[0] == 'domain':
                d.append({'acc':id_,'mdSAM':'+','accession':pd.DataFrame([x.iloc[i].results['metadata']])['accession'].values[0]})
interpro_df = pd.DataFrame(d)

In [ ]:
min_number = 0

In [ ]:
total_runners =  len(G_plot_PM) # No. of hTMs Nodes in the Graph 
number_team_A = len(G_plot_PM[G_plot_PM.acc.isin(interpro_df[interpro_df.accession=='IPR005225'].acc)]) # number mdSAM nodes in the Graph  
number_winners = len(G_plot_PM[(G_plot_PM.obs_exp>0)])## No. of PPIs i.e Edges 
number_win_from_A = len(G_plot_PM[(G_plot_PM.obs_exp>0)&(G_plot_PM.acc.isin(interpro_df[interpro_df.accession=='IPR005225'].acc))]) ## PPIs that are mdSAM
print(total_runners,number_team_A,number_winners,number_win_from_A)
expected = (number_team_A*number_winners)/total_runners
p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
print(1 - p)

In [ ]:
endo_acc = uniport_annotation[uniport_annotation['Gene Ontology IDs'].str.find('GO:0006897')!=-1].Entry
total_runners =  len(G_plot_PM[(G_plot_PM.no_edges_total>min_number)]) # No. of Non-hTMs Nodes in the Graph 
number_team_A =  len(set.intersection(set(endo_acc),set(G_plot_PM[(G_plot_PM.no_edges_total>min_number)].acc)))# number Nodes that are in Endo-data and in Gragh  
number_winners = len(G_plot_PM[(G_plot_PM.obs_exp>0)&(G_plot_PM.log_p>-1*np.log10(1))&(G_plot_PM.no_edges_total>min_number)])
number_win_from_A = len(G_plot_PM[(G_plot_PM.obs_exp>0)&(G_plot_PM.log_p>-1*np.log10(1))&(G_plot_PM.acc.isin(endo_acc))&(G_plot_PM.no_edges_total>min_number)]) ## PPIs that are mdSAM
#print(total_runners,number_team_A,number_winners,number_win_from_A)

expected = (number_team_A*number_winners)/total_runners
print(total_runners,number_team_A,number_winners,number_win_from_A,expected)
p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
print(1 - p)

In [ ]:
G_plot_PM[(G_plot_PM.obs_exp>=3)&(G_plot_PM.log_p>-1*np.log10(.05))]

In [ ]:
len(uniport_annotation[uniport_annotation['Keyword ID'].str.find('KW-0254')!=-1].Entry)

In [ ]:
ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#BCBEC0","#B9DBF4","#155289"])#D55273
ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#95A3A4","#2D5D7B","#DB2B39"])#D55273
#G_plot_PM_shuffel = G2df(G_PM,mdSAMs=pd.Series(list(PM)).sample(len(mdSAM_PM)))
fig, ax = plt.subplots(figsize=(6.5,6))
ax.locator_params(nbins=8)
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
sns.scatterplot(x='obs_exp',y='log_p',ax=ax,
                data=G_plot_PM[G_plot_PM.no_edges_total>min_number],rasterized=False,
                hue='no_edges_total',edgecolor=None,palette=ww,s=75,zorder=2,alpha=1)
sns.scatterplot(x='obs_exp',y='log_p',
                data=G_plot_PM[(G_plot_PM.acc.isin(interpro_df[interpro_df.accession=='IPR005225'].acc))&(G_plot_PM.no_edges_total>min_number)],rasterized=False
                ,edgecolor='k',c='w',s=300,zorder=1,ax=ax)
sns.scatterplot(x='obs_exp',y='log_p',
                data=G_plot_PM[G_plot_PM.acc.isin(endo_acc)],rasterized=False
                ,edgecolor='k',c='#FDAF11',s=300,zorder=1,ax=ax)


ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
norm = Normalize(vmax=G_plot_PM[G_plot_PM.no_edges_total>min_number].loc[:,'no_edges_total'].max()
                 ,vmin=G_plot_PM[G_plot_PM.no_edges_total>min_number].loc[:,'no_edges_total'].min())#plt.Normalize(7, 12)
#sm = plt.cm.ScalarMappable(cmap=ww, norm=norm)
sm = plt.cm.ScalarMappable(cmap=ww, norm=norm)
sm.set_array([])
plt.legend([],frameon=False)
cbar = ax.figure.colorbar(sm,shrink=.30)
cbar.ax.tick_params(labelsize=14)
ax.grid(zorder=0)
cbar.set_label(label='$Total\;No.\,edges$', fontsize=14) 
ax.set_ylabel('$‑log10(p‑val)$',fontsize=18)
ax.set_xlabel('$deg(Obs)-deg(Exp)$\n$Red\;edges$',fontsize=18)
# plt.axvline(3,ls='--',color='k',lw=1)
# plt.axhline(-1*np.log10(.05),ls='--',color='k',lw=1)
#plt.savefig('Rho_enrich_PM_v6.svg',bbox_inches='tight',transparent=True,dpi=600)

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
sns.set(font_scale=1.75,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
sns.scatterplot(x='obs_exp',y='log_p',
                data=G_plot_PM[(G_plot_PM.obs_exp>=2)&(G_plot_PM.log_p>-1*np.log10(.25))],rasterized=False,
                hue='no_edges_total',edgecolor=None,palette=ww,s=200,zorder=2,alpha=1)
sns.scatterplot(x='obs_exp',y='log_p',
                data=G_plot_PM[(G_plot_PM.acc.isin(interpro_df[interpro_df.accession=='IPR005225'].acc))&(G_plot_PM.obs_exp>=2)&(G_plot_PM.log_p>-1*np.log10(.25))],rasterized=False
                ,edgecolor='k',c='w',s=500,zorder=0,ax=ax)
sns.scatterplot(x='obs_exp',y='log_p',
                data=G_plot_PM[(G_plot_PM.acc.isin(endo_acc))&(G_plot_PM.obs_exp>=2)&(G_plot_PM.log_p>-1*np.log10(.25))],rasterized=False
                ,edgecolor='k',c='#FDAF11',s=500,zorder=0,ax=ax)


norm = Normalize(vmax=G_plot_PM.loc[:,'no_edges_total'].max()
                 ,vmin=G_plot_PM.loc[:,'no_edges_total'].min())#plt.Normalize(7, 12)
ax.locator_params(nbins=6)
ax.grid()
texts = []
for i in G_plot_PM[(G_plot_PM.obs_exp>=2)&(G_plot_PM.log_p>-1*np.log10(.5))].index:
    texts.append(plt.text(G_plot_PM.loc[i,'obs_exp'],G_plot_PM.loc[i,'log_p'],
            '$%s$'%G_plot_PM.loc[i,'gene'],
                          fontsize=14))
adjust_text(texts, autoalign='xy',
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)#,force_text=(-.2, .1),force_points=(-.1, 0.1), only_move={'points':'xy', 'text':'xy'},
ax.set_yticklabels(ax.get_yticklabels(),fontsize=24)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=24)
ax.set_ylabel('$‑log10(p‑val)$',fontsize=24)
plt.legend([],frameon=False)
plt.axhline(-1*np.log10(.05),ls='--',color='k',lw=2)
ax.set_xlabel('$deg(Obs)-deg(Exp)$\n$Red\;edges$',fontsize=24)
plt.savefig('Rho_enrich_PM_v6_sig.svg',bbox_inches='tight',transparent=True,dpi=600)

In [ ]:
PM_all = set(uniport_annotation[uniport_annotation['Gene Ontology IDs'].str.find('GO:0005886')!=-1].Entry)# PM GO 
PM_TMs = set.intersection(set(PM_all),set(hTMs.acc))
OD = set(uniport_annotation[uniport_annotation['Gene Ontology IDs'].str.find('GO:0004984')!=-1].Entry)#GPRC Ordarat Receptors GO
TMs_no_OD = set.difference(PM_TMs,OD)
non_TMs_PM = set.intersection(non_TMs,set.difference(PM_all,PM_TMs))

In [ ]:
pd.DataFrame(non_TMs_PM).to_clipboard(index=0)

In [ ]:
G_PM_All = Graph_organelle(organelle=PM_TMs,min_egde=10,max_edge=200,non_TMs=non_TMs_PM)#10,200
mdSAM_PM_all  = set.intersection(set(G_PM_All.nodes),mdSAMs)
G_plot_PM_all = G2df(G_PM_All,mdSAMs=mdSAM_PM_all)

In [ ]:
# pd.DataFrame(NOT_PM).to_clipboard(index=0)

In [ ]:
set.intersection(set(PM_TMs),set(mdSAM_PM_all))

In [ ]:
NOT_PM = set.difference(set(hTMs.acc),PM_TMs)
#NOT_PM = set.difference(set(hTMs.acc),TMs_no_OD)
PM_TMs_NO_OD = set.difference(set(hTMs.acc),TMs_no_OD)

In [ ]:
from scipy.stats import fisher_exact
from scipy.stats.contingency import expected_freq
a = len(uniport_annotation[(uniport_annotation.Entry.isin(mdSAMs))&(uniport_annotation.Entry.isin(PM_TMs))])
b = len(uniport_annotation[(uniport_annotation.Entry.isin(mdSAMs))&(uniport_annotation.Entry.isin(NOT_PM))])
c = len(uniport_annotation[(~uniport_annotation.Entry.isin(mdSAMs))&(uniport_annotation.Entry.isin(PM_TMs))])
d = len(uniport_annotation[(~uniport_annotation.Entry.isin(mdSAMs))&(uniport_annotation.Entry.isin(NOT_PM))])
table= [[a,b],[c,d]]
print(table)
print(expected_freq(table))
enrich = (table-expected_freq(table))[0][0]
    
print(enrich, stats.fisher_exact(table,alternative='two-sided')[-1])

In [ ]:
endo_acc = uniport_annotation[uniport_annotation['Gene Ontology IDs'].str.find('GO:0006897')!=-1].Entry
#G_plot_PM_test =G_plot_PM[G_plot_PM_all.no_edges_total>1]
a = len(G_plot_PM_all[(G_plot_PM_all.obs_exp>8)&(G_plot_PM_all.acc.isin(endo_acc))])
b = len(G_plot_PM_all[(G_plot_PM_all.obs_exp<=8)&(G_plot_PM_all.acc.isin(endo_acc))])
c = len(G_plot_PM_all[(G_plot_PM_all.obs_exp>8)&(~G_plot_PM_all.acc.isin(endo_acc))])
d = len(G_plot_PM_all[(G_plot_PM_all.obs_exp<=8)&(~G_plot_PM_all.acc.isin(endo_acc))])
table_endo_GO= [[a,b],[c,d]]
expected_freq_endo = expected_freq(table_endo_GO)
print(stats.fisher_exact(table_endo_GO,alternative='two-sided')[-1])
# l=[]
# l.append({'val':table[0][0],'ele':'Obs','Box':'A'})
# l.append({'val':expected_freq(table)[0][0],'ele':'Exp','Box':'A'})
# sns.barplot(x='Box',y='val',hue='ele',data=pd.DataFrame(l))
#print(table_endo_GO[0][0]/expected_freq_endo[0][0])



rho_acc = interpro_df[interpro_df.accession=='IPR005225'].acc
#G_plot_PM_test =G_plot_PM[G_plot_PM.no_edges_total>1]
a = len(G_plot_PM_all[(G_plot_PM_all.obs_exp>8)&(G_plot_PM_all.acc.isin(rho_acc))])
b = len(G_plot_PM_all[(G_plot_PM_all.obs_exp<=8)&(G_plot_PM_all.acc.isin(rho_acc))])
c = len(G_plot_PM_all[(G_plot_PM_all.obs_exp>8)&(~G_plot_PM_all.acc.isin(rho_acc))])
d = len(G_plot_PM_all[(G_plot_PM_all.obs_exp<=8)&(~G_plot_PM_all.acc.isin(rho_acc))])
table_rho= [[a,b],[c,d]]

expected_freq_rho = expected_freq(table_rho)
    
print(stats.fisher_exact(table_rho,alternative='two-sided')[-1])
print(expected_freq_rho)
# l=[]
# l.append({'val':table[0][0],'ele':'Obs','Box':'A'})
# l.append({'val':expected_freq(table)[0][0],'ele':'Exp','Box':'A'})


In [ ]:
G_plot_PM_all.acc.to_clipboard(index=0)

In [ ]:
pm_keywords = pd.read_csv('/Users/aelazar/Downloads/idmapping_2023_12_25.tsv',delimiter='\t')

In [ ]:
pm_keywords[pm_keywords['Keyword ID'].str.find('KW-0813')!=-1]

In [ ]:
pm_keys = [] 
for i in pm_keywords.index:
    for j in [i.replace(' ','') for i in pm_keywords.loc[i,'Keyword ID'].split(';')]:
      pm_keys.append({'acc':pm_keywords.loc[i,'Entry'],'ID':j})  
pm_keys_df  = pd.DataFrame(pm_keys)

In [ ]:
pm_keys_df.ID.value_counts().reset_index()

In [ ]:
ps  = []
for ind in  set(pm_keys_df.ID):
    accs = pm_keys_df[pm_keys_df.ID==ind].acc
    #accs = hTMs_uniprot[hTMs_uniprot['Keyword ID'].str.find('KW-0829')!=-1].Entry
    #G_plot_PM_test =G_plot_PM[G_plot_PM.no_edges_total>1]
    GG_plot_sig = G_plot_PM_all[(G_plot_PM_all.p>0)]
    if len(GG_plot_sig[GG_plot_sig.acc.isin(accs)])>20:
        a = len(GG_plot_sig[(GG_plot_sig.obs_exp>0)&(GG_plot_sig.acc.isin(accs))])
        b = len(GG_plot_sig[(GG_plot_sig.obs_exp<=0)&(GG_plot_sig.acc.isin(accs))])
        c = len(GG_plot_sig[(GG_plot_sig.obs_exp>0)&(~GG_plot_sig.acc.isin(accs))])
        d = len(GG_plot_sig[(GG_plot_sig.obs_exp<=0)&(~GG_plot_sig.acc.isin(accs))])
        table_key= [[a,b],[c,d]]
        
        expected_freq_key = expected_freq(table_key)
            
        #print(stats.fisher_exact(table_key,alternative='two-sided')[-1])
        #print(table_key,expected_freq_key)
        ps.append({'ID':ind,'p_val':stats.fisher_exact(table_key,alternative='two-sided')[-1],
                   'table':table_key,'expected_table':expected_freq(table_key),
                   'enrich':np.log2(table_key[0][0]/expected_freq_key[0][0])})

In [ ]:
ps_df = pd.DataFrame(ps)
ps_df['fdr'] = fdr(ps_df['p_val'])
ps_df['fdr_log'] = -1*np.log(ps_df['fdr'])
ps_df.sort_values(by='fdr_log')
ps_df['des'] = ps_df['ID'].map(KeyWord_df.set_index('Keyword ID').to_dict()['Name'])
#ps_df.sort_values(by='enrich').tail(10).loc[:,['des','enrich','fdr_log']]
fig, ax = plt.subplots(figsize=(3,3))
sns.scatterplot(x='enrich',y='fdr_log',data=ps_df)
plt.yscale('symlog')
plt.axhline(.6)

In [ ]:
G_plot_PM_all.to_excel('G_plot_Sol_PM.xlsx')

In [ ]:
 G_plot_PM_all.no_edges.sum()

In [ ]:
len(G_plot_PM_all[G_plot_PM_all.obs_exp<0])/len(G_plot_PM_all)

In [ ]:
from matplotlib.colors import LogNorm, Normalize

In [ ]:
norm = Normalize(vmax=G_plot_PM_all[G_plot_PM_all.no_edges_total>min_number].loc[:,'no_edges_total'].max()
                 ,vmin=G_plot_PM_all[G_plot_PM_all.no_edges_total>min_number].loc[:,'no_edges_total'].min())#p
fig, axs = plt.subplot_mosaic("AB;AC",gridspec_kw = {'hspace':.2,'wspace':.1,'width_ratios': [2,1]},figsize = (7,5))
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=G_plot_PM_all[G_plot_PM_all.no_edges_total>min_number],rasterized=False,s=75,
                hue='no_edges_total',edgecolor=None,palette=ww,zorder=2,alpha=1,sizes='no_edges_total',ax=axs['A'])
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=G_plot_PM_all[(G_plot_PM_all.acc.isin(interpro_df[interpro_df.accession=='IPR005225'].acc))],rasterized=False
                ,edgecolor='k',c='k',s=30,zorder=2,ax=axs['B'])
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=G_plot_PM_all[(~G_plot_PM_all.acc.isin(interpro_df[interpro_df.accession=='IPR005225'].acc))],rasterized=False
                ,edgecolor=None,c='gray',s=30,zorder=1,ax=axs['B'])
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=G_plot_PM_all[G_plot_PM_all.acc.isin(endo_acc)],rasterized=False
                ,edgecolor=None,c='#FDAF11',s=30,zorder=2,ax=axs['C'])
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=G_plot_PM_all[~G_plot_PM_all.acc.isin(endo_acc)],rasterized=False
                ,edgecolor=None,c='gray',s=30,zorder=1,ax=axs['C'])
for i in ['A','B','C']:
    axs[i].grid(zorder=0)
    axs[i].set_xlim([-10,21])
    axs[i].set_ylim([0,2])
    axs[i].locator_params(nbins=6)
    if i!='A':
        axs[i].set_xticklabels([])
        axs[i].set_yticklabels([])
        axs[i].set_xlabel('')
        axs[i].set_ylabel('')
        axs[i].axhline(-1*np.log10(.25),ls='--',color='k',lw=2)
        axs[i].axvline(8,ls='--',color='k',lw=2)
axs['A'].set_yticklabels(axs['A'].get_yticklabels(),fontsize=18)
axs['A'].set_xticklabels(axs['A'].get_xticklabels(),fontsize=18)
axs['A'].set_ylabel('$‑log10(FDR)$',fontsize=20)
h,l = axs['A'].get_legend_handles_labels()
axs['A'].legend(h[:], l[:],title='Total No edges',frameon=True,fontsize=10
               )   # <<<<<<<< This is where the magic happens

axs['B'].set_title('Small GTP bd dom',fontsize=14)
axs['B'].axis('off')
axs['C'].axis('off')
axs['C'].set_title('GO:Endocytosis',fontsize=14)
axs['A'].axhline(-1*np.log10(.25),ls='--',color='k',lw=2)
axs['A'].axvline(8,ls='--',color='k',lw=2)

#axs['A'].set_xlabel('$deg(Obs)-deg(Exp)$\n$Red\;edges$',fontsize=20)
axs['A'].set_xlabel('$Obs[\;\;\;]-Exp[\;\;\;]$',fontsize=20)

axs['B'].text(10,1.7,'%d/%d'%(table_rho[0][0],expected_freq_rho[0][0]),fontsize=18)
axs['C'].text(10,1.7,'%d/%d'%(table_endo_GO[0][0],expected_freq_endo[0][0]),fontsize=18)
texts = []
for i in G_plot_PM_all[(G_plot_PM_all.obs_exp>=8)&(G_plot_PM_all.log_fdr>-1*np.log10(.5))].index:
    texts.append(axs['A'].text(G_plot_PM_all.loc[i,'obs_exp'],G_plot_PM_all.loc[i,'log_fdr'],
            '$%s$'%G_plot_PM_all.loc[i,'gene'],
                          fontsize=12))
adjust_text(texts, autoalign='xy',ax=axs['A'],
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
#plt.savefig('Rho_enrich_PM_v7.svg',bbox_inches='tight',transparent=True,dpi=600)
norm = Normalize(vmax=200,vmin=1)
sm = plt.cm.ScalarMappable(cmap=ww, norm=norm)
sm.set_array([])
cbar = axs['A'].figure.colorbar(sm,shrink=.55)
cbar.ax.tick_params(labelsize=12)
cbar.set_label(label='Total No edges', size='large')
plt.savefig('Rho_enrich_PM_v8.svg',bbox_inches='tight',transparent=True,dpi=600)


In [ ]:
G_plot_PM_all

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=G_plot_PM_all[G_plot_PM_all.no_edges_total>min_number],rasterized=False,s=25,
                edgecolor=None,color='gray',zorder=2,alpha=.8,sizes='no_edges_total',ax=ax)
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=G_plot_PM_all[(G_plot_PM_all.no_edges_total>min_number)&(G_plot_PM_all.obs_exp>8)&(G_plot_PM_all.log_fdr>-1*np.log10(.25))],rasterized=False,s=50,
                edgecolor=None,color='#6F619B',zorder=2,alpha=1,sizes='no_edges_total',ax=ax)
plt.legend([],frameon=False)
plt.xlim([-10,25])
plt.ylim([0,2])
# texts = []
# for i in G_plot_PM_all[(G_plot_PM_all.obs_exp>=8)&(G_plot_PM_all.log_fdr>-1*np.log10(.25))].index:
#     texts.append(ax.text(G_plot_PM_all.loc[i,'obs_exp'],G_plot_PM_all.loc[i,'log_fdr'],
#             '$%s$'%G_plot_PM_all.loc[i,'gene'],
#                           fontsize=12))
# adjust_text(texts, autoalign='y',ax=ax,force_text=(0, 1),
#             arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
ax.grid()
plt.locator_params(axis='both',nbins=4)
plt.ylabel('FDR(-log$_{10}$)',fontsize=12)
plt.xlabel('Obs-Exp',fontsize=12)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.axvline(8,color='k',ls='--',zorder=0)
plt.axhline(-1*np.log10(.25),color='k',ls='--',zorder=0)
plt.savefig('Rho_enrich_PM_v9.svg',bbox_inches='tight',transparent=True,dpi=600)


In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=G_plot_PM_all[(G_plot_PM_all.acc.isin(interpro_df[interpro_df.accession=='IPR005225'].acc))],rasterized=False
                ,edgecolor='k',c='k',s=50,zorder=2,ax=ax)
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=G_plot_PM_all[(~G_plot_PM_all.acc.isin(interpro_df[interpro_df.accession=='IPR005225'].acc))],rasterized=False
                ,edgecolor=None,c='gray',s=25,zorder=1,ax=ax)
plt.legend([],frameon=False)
plt.xlim([-10,25])
plt.ylim([0,2])
plt.ylabel('FDR(-log$_{10}$)',fontsize=12)
plt.xlabel('Obs-Exp',fontsize=12)
plt.axvline(8,color='k',ls='--',zorder=0)
plt.axhline(-1*np.log10(.25),color='k',ls='--',zorder=0)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
ax.grid()
plt.locator_params(axis='both',nbins=4)
# ax.text(10,1.7,'[%d/%d]'%(table_rho[0][0],expected_freq_rho[0][0]),fontsize=24)
#ax.set_title('Small GTP bd dom',fontsize=16)
# texts = []
# for i in G_plot_PM_all[(G_plot_PM_all.obs_exp>=8)&(G_plot_PM_all.log_fdr>-1*np.log10(.25))].index:
#     texts.append(ax.text(G_plot_PM_all.loc[i,'obs_exp'],G_plot_PM_all.loc[i,'log_fdr'],
#             '$%s$'%G_plot_PM_all.loc[i,'gene'],
#                           fontsize=12))
# adjust_text(texts, autoalign='xy',ax=ax,
#             arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)


plt.savefig('Rho_enrich_PM_GTPase.svg',bbox_inches='tight',transparent=True,dpi=600)


In [ ]:
G_plot_PM_all[(G_plot_PM_all.acc.isin(interpro_df[interpro_df.accession=='IPR005225'].acc))
&(G_plot_PM_all.log_fdr>.6)]

In [ ]:
mdSAM_loc = pd.read_pickle('mdSAM_loacation.pkl')
Extracellular= set(mdSAM_loc[mdSAM_loc.location.isin(['Lumenal','Extracellular'])].acc)
Cytoplasmic= set(mdSAM_loc[mdSAM_loc.location=='Cytoplasmic'].acc)
both = (set.intersection(Extracellular,Cytoplasmic))
Cytoplasmic_only = set.difference(Cytoplasmic,both)
Extracellular_only = set.difference(Extracellular,both)
list(G_PM_All['Q96MU7'])

In [ ]:
def G_hypergeom(G,acc,mdSAMs=mdSAM_PM):
    total_runners =  len(set.intersection(set(list(G.nodes)),set(sub_TMs))) # No. of hTMs Nodes in the Graph 
    number_team_A = len(set.intersection(set(list(G.nodes)),mdSAMs)) # number mdSAM nodes in the Graph  
    number_winners = len(list(G[acc]))## No. of PPIs i.e Edges 
    number_win_from_A = len(set.intersection(set(list(G[acc])),mdSAMs)) ## PPIs that are mdSAM
    #print(total_runners,number_team_A,number_winners,number_win_from_A)
    expected = (number_team_A*number_winners)/total_runners
    p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
    #print(p)
    if  number_win_from_A-expected >0:
        p = 1-p 
    return number_win_from_A,expected,p 

In [ ]:
G_hypergeom(G_PM_All,'P63000',mdSAMs=Extracellular_only)

In [ ]:
b= (len(set.intersection(set(list(G_PM_All['Q96L33'])),Cytoplasmic_only)))
a= (len(set.intersection(set(list(G_PM_All['Q96L33'])),Extracellular_only)))
b,a

In [ ]:
mdSAMs

In [ ]:
z = []
for i  in set(G_PM_All.nodes):
    z.append(i[0])
    z.append(i[1])

In [ ]:
e = set(list(G_PM_All.nodes))

In [ ]:
len(set.intersection(e,Cytoplasmic_only))

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=G_plot_PM_all[G_plot_PM_all.acc.isin(endo_acc)],rasterized=False
                ,edgecolor=None,c='#F2B342',s=50,zorder=2,ax=ax)
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=G_plot_PM_all[~G_plot_PM_all.acc.isin(endo_acc)],rasterized=False
                ,edgecolor=None,c='gray',s=25,zorder=1,ax=ax)
plt.legend([],frameon=False)
plt.xlim([-10,25])
plt.ylim([0,2])
plt.ylabel('FDR(-log$_{10}$)',fontsize=12)
plt.xlabel('Obs-Exp',fontsize=12)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.axvline(8,color='k',ls='--',zorder=0)
plt.axhline(-1*np.log10(.25),color='k',ls='--',zorder=0)
ax.grid()
plt.locator_params(axis='both',nbins=4)
#ax.text(10,1.7,'[%d/%d]'%(table_endo_GO[0][0],expected_freq_endo[0][0]),fontsize=24)#ax.set_title('Small GTP bd dom',fontsize=20)
# texts = []
# for i in G_plot_PM_all[(G_plot_PM_all.obs_exp>=8)&(G_plot_PM_all.log_fdr>-1*np.log10(.25))].index:
#     texts.append(ax.text(G_plot_PM_all.loc[i,'obs_exp'],G_plot_PM_all.loc[i,'log_fdr'],
#             '$%s$'%G_plot_PM_all.loc[i,'gene'],
#                           fontsize=12))
# adjust_text(texts, autoalign='xy',ax=ax,
#             arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)

#ax.set_title('GO:Endocytosis',fontsize=20)
plt.savefig('Rho_enrich_PM_Endo.svg',bbox_inches='tight',transparent=True,dpi=600)


In [ ]:
print('%d mdSAM nodes'%len(set.intersection(set(G_PM_All.nodes),mdSAM_PM_all)))
print('%d non-mdSAM nodes'%len(set.intersection(set(G_PM_All.nodes),set.difference(PM_all,mdSAM_PM_all))))
print('%d non-TM nodes'%len(set.intersection(set(G_PM_All.nodes),non_TMs)))


In [ ]:
G_PM_All

In [ ]:
d = []
for i in G_PM_All.degree:
    if i[1]>0:
        if i[0] in mdSAMs:
            d.append({'acc':i[0],'degree':i[1],'mdSAM':'+'})
        else:
            d.append({'acc':i[0],'degree':i[1],'mdSAM':'-'})
dd = pd.DataFrame(d)
dd['box']='A'
stats.ttest_ind(dd[dd.mdSAM=='+'].degree,
              dd[dd.mdSAM=='-'].degree)
d3 = dd[dd.degree.between(1,1000)]
fig, ax = plt.subplots(figsize=(3,6)) 
sns.boxplot(x='mdSAM',y='degree',data=d3
                  ,showfliers=False,zorder=2,
                 fliersize=0, width=0.9, medianprops=dict(color="k", alpha=1,zorder=2,linewidth=1), boxprops=dict(linewidth=1, facecolor='w', edgecolor='k', alpha=.61), 
                 whiskerprops=dict(linewidth=1, color='k', alpha=.6,zorder=3.75), capprops=dict(linewidth=1, color='k', alpha=.6,zorder=3),native_scale=True)
sns.swarmplot(x='mdSAM',y='degree',data=d3,s=2,zorder=2,dodge=False,palette=['#B4B5B4','#C03830'],hue='mdSAM')
plt.ylabel('Degree',fontsize=16)
plt.xlabel('mdSAM',fontsize=16)
ax.grid()
ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
plt.savefig('Dgree_Sol-MP.svg',bbox_inches='tight',transparent=True,dpi=600)

In [ ]:
MP_Sol_deg = d3
sns.ecdfplot(hue='mdSAM',x='degree',data=MP_Sol_deg)

stats.ks_2samp(dd[dd.mdSAM=='+'].degree,
              dd[dd.mdSAM=='-'].degree)

In [ ]:
fig, ax = plt.subplots(figsize=(3,4))
sns.violinplot(hue='mdSAM',y='degree',data=MP_Sol_deg,split=1,fill=0,hue_order=['-','+'],palette=['gray','r'])

In [ ]:
MP_Sol_deg[MP_Sol_deg.mdSAM=='+'].degree.median()

In [ ]:
print('%d mdSAM nodes'%len(set.intersection(set(G_PM_All.nodes),mdSAM_PM_all)))
print('%d non-mdSAM nodes'%len(set.intersection(set(G_PM_All.nodes),set.difference(PM_all,mdSAM_PM_all))))
print('%d non-TM nodes'%len(set.intersection(set(G_PM_All.nodes),non_TMs_PM)))


In [ ]:
endo_data = pd.read_excel('/Users/aelazar/Downloads/41586_2010_BFnature08779_MOESM228_ESM.xlsx',
                          sheet_name=' 4609 genes with score values',header=1)
id2acc = pd.read_csv('/Users/aelazar/notebooks_for_paper/endo_dataset/idmapping_reviewed_true_2023_06_29.tsv',delimiter='\t')
for i in endo_data.index:
    if id2acc[id2acc.From==endo_data.loc[i,'Gene ID']].empty ==False:
        endo_data.loc[i,'acc'] = id2acc[id2acc.From==endo_data.loc[i,'Gene ID']].Entry.values[0]

In [ ]:
endo_et_al = endo_data[endo_data.acc.isin(non_TMs_PM)]

In [ ]:
len(G_plot_PM_all[(G_plot_PM_all.obs_exp>5)&(G_plot_PM_all.acc.isin(endo_et_al.acc))&(G_plot_PM_all.log_fdr>0.6)])

In [ ]:
total_runners =  len(set(G_plot_PM_all.acc)) # No. of hTMs Nodes in the Graph 
number_team_A = len(set.intersection(set(G_plot_PM_all.acc),set(endo_et_al.acc))) # number mdSAM nodes in the Graph  
number_winners = len(G_plot_PM_all[(G_plot_PM_all.obs_exp>2)&(G_plot_PM_all.log_fdr>0.6)])
number_win_from_A = len(G_plot_PM_all[(G_plot_PM_all.obs_exp>2)&(G_plot_PM_all.log_fdr>0.6)&(G_plot_PM_all.acc.isin(endo_et_al.acc))])
expected = (number_team_A*number_winners)/total_runners
print(total_runners,number_team_A,number_winners,number_win_from_A,expected)
p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
print(1-p)

In [ ]:
len(PM_TMs)

In [ ]:
mdSAMs_PM = set.intersection(PM_TMs,mdSAMs)

In [ ]:
GG = nx.Graph()
l = []
#accs = (set.union(set(df.acc1),set(df.acc2))) ## Nodes 
for acc in PM_TMs:#nodes_for_graph:
    if acc in list(mdSAMs_PM):
        l.append((acc, {"mdSAM": "yes"}))
    if acc in list(set.difference(set(PM_TMs),mdSAMs_PM)):
        l.append((acc, {"mdSAM": "no"}))
GG.add_nodes_from(l)
for ind in df.index:
    if ((df.loc[ind,'acc1'] in PM_TMs) and (df.loc[ind,'acc2'] in PM_TMs)):
        if df.loc[ind,'acc1']!=df.loc[ind,'acc2']:
            GG.add_edge(df.loc[ind,'acc1'], df.loc[ind,'acc2'])
z = []
for i in GG.nodes:
    if len(GG[i])==0:
        z.append(i)
        #GG.remove_node(i)
GG.remove_nodes_from(z)

In [ ]:
sns.ecdfplot([i[1] for i in GG.degree])

In [ ]:
# import scipy.stats as stats
# import numpy as np
# import matplotlib.pyplot as plt

# # Count number of "yes" nodes
# n_yes = sum(1 for node in GG.nodes if GG.nodes[node]["mdSAM"] == "yes")
# n_nodes = GG.number_of_nodes()
# p_yes = n_yes / n_nodes

# # Count edges
# count_yes_yes = 0
# total_edges = 0

# for u, v in GG.edges():
#     if u == v:
#         continue  # skip self-loops
#     u_mdSAM = GG.nodes[u]["mdSAM"]
#     v_mdSAM = GG.nodes[v]["mdSAM"]
    
#     total_edges += 1
#     if (u_mdSAM == "yes") and (v_mdSAM == "yes"):
#         count_yes_yes += 1

# # Expected probability and expected count
# p_expected_yes_yes = p_yes * p_yes
# expected_yes_yes_edges = total_edges * p_expected_yes_yes

# # Perform binomial test
# result = stats.binomtest(count_yes_yes, total_edges, p_expected_yes_yes, alternative='less')

# # Print numbers
# print(f"Observed yes-yes edges: {count_yes_yes}")
# print(f"Total edges: {total_edges}")
# print(f"Expected yes-yes probability: {p_expected_yes_yes:.4f}")
# print(f"Expected number of yes-yes edges: {expected_yes_yes_edges:.2f}")
# print(f"Binomial test p-value: {result.pvalue:.4f}")

# # --- Plot PMF ---
# k = np.arange(0, total_edges + 1)  # all possible counts
# pmf = stats.binom.pmf(k, total_edges, p_expected_yes_yes)
# plt.figure(figsize=(3,3))

# plt.plot(k, pmf, color='black', lw=2)

# # Expected line
# plt.axvline(expected_yes_yes_edges, color='orange', linestyle='--', lw=2)
# plt.text(expected_yes_yes_edges + 1, max(pmf)*0.9, f"Expected\n({expected_yes_yes_edges:.1f})", color='orange', ha='left', va='center', fontsize=10)

# # Observed line
# plt.axvline(count_yes_yes, color='brown', linestyle='-', lw=2)
# plt.text(count_yes_yes + 1, max(pmf)*0.6, f"Observed\n({count_yes_yes})", color='brown', ha='left', va='center', fontsize=10)

# # Labels and style
# plt.title('Binomial PMF of Yes-Yes Edges', fontsize=14)
# plt.xlabel('Number of Yes-Yes Edges')
# plt.ylabel('PMF', fontsize=12)
# plt.xlim([650,900])
# plt.tight_layout()



In [ ]:
from scipy.stats import fisher_exact

# Total possible edges (excluding self-loops)
M = n_nodes * (n_nodes - 1) // 2

# Observed
total_edges = GG.number_of_edges()

# Possible edges of each type
possible_yes_yes = n_yes * (n_yes - 1) // 2
possible_no_no = n_no * (n_no - 1) // 2
possible_yes_no = n_yes * n_no

# 1. Fisher's Exact for yes-yes
table_yes_yes = [
    [count_yes_yes, total_edges - count_yes_yes],
    [possible_yes_yes, M - possible_yes_yes]
]
oddsratio_yy, p_yy = fisher_exact(table_yes_yes, alternative='less')

# 2. Fisher's Exact for no-no
table_no_no = [
    [count_no_no, total_edges - count_no_no],
    [possible_no_no, M - possible_no_no]
]
oddsratio_nn, p_nn = fisher_exact(table_no_no, alternative='greater')

# 3. Fisher's Exact for yes-no
table_yes_no = [
    [count_yes_no, total_edges - count_yes_no],
    [possible_yes_no, M - possible_yes_no]
]
oddsratio_yn, p_yn = fisher_exact(table_yes_no, alternative='less')

# Print results
print("Fisher's Exact Test Results:")
print(f"Yes-Yes: p = {p_yy:.4e}, odds ratio = {oddsratio_yy:.2f}")
print(f"No-No:   p = {p_nn:.4e}, odds ratio = {oddsratio_nn:.2f}")
print(f"Yes-No:  p = {p_yn:.4e}, odds ratio = {oddsratio_yn:.2f}")


In [ ]:
oddsratio_yn

In [ ]:

# Replace these with your actual computed values
data = {
    'Edge Type': ['yes-yes', 'no-no', 'yes-no'],
    'exp': [expected_freq(table_yes_yes)[0][0], expected_freq(table_no_no)[0][0], expected_freq(table_yes_no)[0][0]],
    'obs': [table_yes_yes[0][0], table_no_no[0][0], table_yes_no[0][0]],
    'odds_ratio':[oddsratio_yy,oddsratio_nn,oddsratio_yn]
}
plot_df_odds = pd.DataFrame(data)
#plot_df_odds.set_index('Edge Type', inplace=True)
plot_df_odds

In [ ]:
# Replace these with your actual computed values
data = {
    'Edge Type': ['yes-yes', 'no-no', 'yes-no'],
    'exp': [expected_freq(table_yes_yes)[0][0], expected_freq(table_no_no)[0][0], expected_freq(table_yes_no)[0][0]],
    'obs': [table_yes_yes[0][0], table_no_no[0][0], table_yes_no[0][0]],
    'odds_ratio':[oddsratio_yy,oddsratio_nn,oddsratio_yn]
}
plot_df_odds = pd.DataFrame(data)
#plot_df_odds.set_index('Edge Type', inplace=True)
plot_df_odds
fig, ax = plt.subplots(figsize=(3,3)) 
sns.barplot(y='Edge Type',x='odds_ratio',data=plot_df_odds,order=['no-no','yes-no','yes-yes'])
plt.axvline(1)
plt.locator_params(axis='x',nbins=4)
plt.savefig('Membrane_graph_BioGrid_v6.svg',bbox_inches='tight',transparent=True)

In [ ]:
table_yes_yes

In [ ]:
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt

# Count number of "yes" nodes
n_yes = sum(1 for node in GG.nodes if GG.nodes[node]["mdSAM"] == "yes")
n_nodes = GG.number_of_nodes()
n_no = n_nodes - n_yes

# For hypergeometric calculation
# M = total population size (total possible edges)
# n = number of success states in population (possible yes-yes edges)
# N = sample size (actual edges in graph)
# k = number of observed successes (actual yes-yes edges)

# Total possible edges in a complete graph (excluding self-loops)
M = n_nodes * (n_nodes - 1)  

# Number of possible yes-yes edges
n = n_yes * (n_yes - 1)

# Actual number of edges
N = GG.number_of_edges()

# Count yes-yes edges
count_yes_yes = 0
for u, v in GG.edges():
    if u == v:
        continue  # skip self-loops
    if GG.nodes[u]["mdSAM"] == "yes" and GG.nodes[v]["mdSAM"] == "yes":
        count_yes_yes += 1

# Expected count under random connection model
expected_yes_yes_edges = (n / M) * N

# Perform hypergeometric test
# We use 'less' because we're testing if there are fewer yes-yes edges than expected
result = stats.hypergeom.sf(count_yes_yes - 1, M, n, N)

# Print numbers
print(f"Yes nodes: {n_yes} out of {n_nodes} nodes ({n_yes/n_nodes:.2%})")
print(f"Observed yes-yes edges: {count_yes_yes}")
print(f"Total edges in graph: {N}")
print(f"Total possible edges: {M}")
print(f"Possible yes-yes edges: {n}")
print(f"Expected yes-yes edges: {expected_yes_yes_edges:.2f}")
print(f"Hypergeometric p-value: {1-result:.4f}")

# --- Plot PMF ---
# Range of possible yes-yes edges
k_range = np.arange(max(0, N - (M - n)), min(n, N) + 1)
pmf = stats.hypergeom.pmf(k_range, M, n, N)

plt.figure(figsize=(3, 3))
plt.plot(k_range, pmf, color='black', lw=2)

# Expected line
plt.axvline(expected_yes_yes_edges, color='orange', linestyle='--', lw=2)
plt.text(expected_yes_yes_edges + 5, max(pmf)*0.9, f"Expected\n({expected_yes_yes_edges:.1f})", 
         color='orange', ha='left', va='center', fontsize=10)

# Observed line
plt.axvline(count_yes_yes, color='brown', linestyle='-', lw=2)
plt.text(count_yes_yes + 5, max(pmf)*0.6, f"Observed\n({count_yes_yes})", 
         color='brown', ha='left', va='center', fontsize=10)

# Labels and style
#plt.title('Hypergeometric PMF of Yes-Yes Edges', fontsize=14)
plt.xlabel('Number of Yes-Yes Edges')
plt.ylabel('PMF', fontsize=12)
plt.locator_params(axis='y',nbins=4)

# For similar x-axis range as original
# Adjust these values based on actual data
expected_range = int(expected_yes_yes_edges * 0.2)
plt.xlim([max(0, expected_yes_yes_edges - expected_range), 
          min(n, expected_yes_yes_edges + expected_range)])
plt.savefig('Membrane_graph_BioGrid_v5.svg',bbox_inches='tight',transparent=True)

In [ ]:
count_yes_yes = 0
for u, v in GG.edges():
    if u == v:
        continue  # skip self-loops
    if GG.nodes[u]["mdSAM"] == "no" and GG.nodes[v]["mdSAM"] == "no":
        count_yes_yes += 1
count_yes_yes

In [ ]:
# Initialize counts
count_yes_yes = 0
count_no_no = 0
count_yes_no = 0

for u, v in GG.edges():
    if u == v:
        continue  # skip self-loops
    label_u = GG.nodes[u]["mdSAM"]
    label_v = GG.nodes[v]["mdSAM"]
    
    if label_u == "yes" and label_v == "yes":
        count_yes_yes += 1
    elif label_u == "no" and label_v == "no":
        count_no_no += 1
    elif (label_u == "yes" and label_v == "no") or (label_u == "no" and label_v == "yes"):
        count_yes_no += 1

# Total edges check
assert count_yes_yes + count_no_no + count_yes_no == GG.number_of_edges(), "Edge count mismatch"

# Print all edge type counts
print(f"Observed yes-yes edges: {count_yes_yes}")
print(f"Observed no-no edges: {count_no_no}")
print(f"Observed yes-no edges: {count_yes_no}")

In [ ]:
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt

# Count number of "no" nodes
n_no = sum(1 for node in GG.nodes if GG.nodes[node]["mdSAM"] == "no")
n_nodes = GG.number_of_nodes()
n_yes = n_nodes - n_no

# For hypergeometric calculation
# M = total population size (total possible edges)
# n = number of success states in population (possible no-no edges)
# N = sample size (actual edges in graph)
# k = number of observed successes (actual no-no edges)

# Total possible edges in a complete graph (excluding self-loops)
M = n_nodes * (n_nodes - 1)

# Number of possible no-no edges
n = n_no * (n_no - 1)

# Actual number of edges
N = GG.number_of_edges()

# Count no-no edges
count_no_no = 0
for u, v in GG.edges():
    if u == v:
        continue  # skip self-loops
    if GG.nodes[u]["mdSAM"] == "no" and GG.nodes[v]["mdSAM"] == "no":
        count_no_no += 1

# Expected count under random connection model
expected_no_no_edges = (n / M) * N

# Perform hypergeometric test
# We use 'greater' because we're testing if there are more no-no edges than expected
result = stats.hypergeom.sf(count_no_no - 1, M, n, N)

# Print numbers
print(f"No nodes: {n_no} out of {n_nodes} nodes ({n_no/n_nodes:.2%})")
print(f"Observed no-no edges: {count_no_no}")
print(f"Total edges in graph: {N}")
print(f"Total possible edges: {M}")
print(f"Possible no-no edges: {n}")
print(f"Expected no-no edges: {expected_no_no_edges:.2f}")
print(f"Hypergeometric p-value: {1-result:.4f}")

# --- Plot PMF ---
k_range = np.arange(max(0, N - (M - n)), min(n, N) + 1)
pmf = stats.hypergeom.pmf(k_range, M, n, N)

plt.figure(figsize=(3, 3))
plt.plot(k_range, pmf, color='black', lw=2)

# Expected line
plt.axvline(expected_no_no_edges, color='orange', linestyle='--', lw=2)
plt.text(expected_no_no_edges + 5, max(pmf)*0.9, f"Expected\n({expected_no_no_edges:.1f})", 
         color='orange', ha='left', va='center', fontsize=10)

# Observed line
plt.axvline(count_no_no, color='blue', linestyle='-', lw=2)
# plt.text(count_no_no + 5, max(pmf)*0.6, f"Observed\n({count_no_no})", 
#          color='blue', ha='left', va='center', fontsize=10)

# Labels and style
#plt.title('Hypergeometric PMF of No-No Edges', fontsize=14)
plt.xlabel('Number of No-No Edges')
plt.ylabel('PMF', fontsize=12)
plt.locator_params(axis='y', nbins=4)

expected_range = int(expected_no_no_edges * 0.1)
plt.xlim([max(0, expected_no_no_edges - expected_range), 
          min(n, expected_no_no_edges + expected_range)])
plt.savefig('Membrane_graph_BioGrid_v5_no_no.svg', bbox_inches='tight', transparent=True)


In [ ]:
# Total possible edges
M = n_nodes * (n_nodes - 1)

# Number of possible yes-no edges
possible_yes_no =2* n_yes * n_no

# Expected number of yes-no edges
expected_yes_no_edges = (possible_yes_no / M) * N

# Perform hypergeometric test for yes-no edges
result_yes_no = stats.hypergeom.sf(count_yes_no - 1, M, possible_yes_no, N)

# Print results
print(f"Observed yes-no edges: {count_yes_no}")
print(f"Possible yes-no edges: {possible_yes_no}")
print(f"Expected yes-no edges: {expected_yes_no_edges:.2f}")
print(f"Hypergeometric p-value: {1 - result_yes_no:.4f}")

In [ ]:
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt

# Count number of "no" and "yes" nodes
n_no = sum(1 for node in GG.nodes if GG.nodes[node]["mdSAM"] == "no")
n_yes = sum(1 for node in GG.nodes if GG.nodes[node]["mdSAM"] == "yes")
n_nodes = n_no + n_yes

# Total possible undirected edges (no self-loops)
M = n_nodes * (n_nodes - 1) // 2

# Number of possible yes-no edges (undirected)
n = n_no * n_yes

# Actual number of edges in graph
N = GG.number_of_edges()

# Count observed yes-no edges
count_yes_no = 0
for u, v in GG.edges():
    if (GG.nodes[u]["mdSAM"] == "no" and GG.nodes[v]["mdSAM"] == "yes") or \
       (GG.nodes[u]["mdSAM"] == "yes" and GG.nodes[v]["mdSAM"] == "no"):
        count_yes_no += 1

# Expected yes-no edges under random model
expected_yes_no_edges = (n / M) * N

# Hypergeometric test (one-sided: enrichment)
p_val = stats.hypergeom.sf(count_yes_no - 1, M, n, N)

# Print summary
print(f"No nodes: {n_no} out of {n_nodes} nodes ({n_no/n_nodes:.2%})")
print(f"Yes nodes: {n_yes} out of {n_nodes} nodes ({n_yes/n_nodes:.2%})")
print(f"Observed yes-no edges: {count_yes_no}")
print(f"Total edges in graph: {N}")
print(f"Total possible edges (M): {M}")
print(f"Possible yes-no edges (n): {n}")
print(f"Expected yes-no edges: {expected_yes_no_edges:.2f}")
print(f"Hypergeometric p-value: {p_val:.4f}")

# --- Plot PMF ---
k_range = np.arange(max(0, N - (M - n)), min(n, N) + 1)
pmf = stats.hypergeom.pmf(k_range, M, n, N)

plt.figure(figsize=(3, 3))
plt.plot(k_range, pmf, color='black', lw=2)

# Expected line
plt.axvline(expected_yes_no_edges, color='orange', linestyle='--', lw=2)
plt.text(expected_yes_no_edges + 1, max(pmf)*0.9, f"Expected\n({expected_yes_no_edges:.1f})", 
         color='orange', ha='left', va='center', fontsize=10)

# Observed line
plt.axvline(count_yes_no, color='blue', linestyle='-', lw=2)
plt.text(count_yes_no + 1, max(pmf)*0.6, f"Observed\n({count_yes_no})", 
         color='blue', ha='left', va='center', fontsize=10)

# Labels and formatting
#plt.title('Hypergeometric PMF of Yes-No Edges', fontsize=14)
plt.xlabel('Number of Yes-No Edges')
plt.ylabel('PMF', fontsize=12)
plt.locator_params(axis='y', nbins=4)

expected_range = max(5, int(expected_yes_no_edges * 0.2))
plt.xlim([max(0, int(expected_yes_no_edges - expected_range)), 
          min(n, int(expected_yes_no_edges + expected_range))])
plt.tight_layout()
plt.savefig('Membrane_graph_BioGrid_v5_yes_no.svg', bbox_inches='tight', transparent=True)


In [ ]:
def GG_hypergeom(acc,mdSAMs):
    total_runners =  len(set(list(GG.nodes))) # No. of hTMs Nodes in the Graph 
    number_team_A = len(set.intersection(set(list(GG.nodes)),mdSAMs)) # number mdSAM nodes in the Graph  
    number_winners = len(list(GG[acc]))## No. of PPIs i.e Edges 
    number_win_from_A = len(set.intersection(set(list(GG[acc])),mdSAMs)) ## PPIs that are mdSAM
    #print(total_runners,number_team_A,number_winners,number_win_from_A)
    expected = (number_team_A*number_winners)/total_runners
    p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
    #print(p)
    if  number_win_from_A-expected >0:
        p = 1-p 
    return number_win_from_A,expected,p 

In [ ]:
d = []
for i in GG.degree:
    if i[1]>0:
        if i[0] in mdSAMs:
            d.append({'acc':i[0],'degree':i[1],'mdSAM':'+'})
        else:
            d.append({'acc':i[0],'degree':i[1],'mdSAM':'-'})
dd = pd.DataFrame(d)
dd['box']='A'

In [ ]:
sns.boxplot(hue='mdSAM',y='degree',data=dd)

In [ ]:
print('%d mdSAM nodes'%len(set.intersection(set(GG.nodes),mdSAMs_PM)))
print('%d non-mdSAM nodes'%len(set.intersection(set(GG.nodes),set.difference(PM_all,mdSAMs_PM))))
print('%d non-TM nodes'%len(set.intersection(set(GG.nodes),non_TMs_PM)))
print('%d non-TM edges'%len(set.intersection(set(GG.nodes),non_TMs_PM)))

In [ ]:
GG_hypergeom_list =[]
for i in dd.acc:
    n,e,p  = GG_hypergeom(i,mdSAMs)#pd.Series(list(PM_TMs)).sample(len(mdSAMs)))
    if acc2name[acc2name.From==i]['Gene Names'].empty==False:
        gene = acc2name[acc2name.From==i]['Gene Names'].values[0].split(' ')[0]
    else:
        gene= np.NaN
    GG_hypergeom_list.append({'acc':i,'no_edges':n,'expected':e,'p':p,'no_edges_total':len(set.intersection(set(list(GG[i])),set(hTMs.acc))),
                             'gene':gene,'round':'real'})
GG_plot = pd.DataFrame(GG_hypergeom_list)
GG_plot['obs_exp'] = (GG_plot['no_edges']-GG_plot['expected'])#/G_hypergeom_df['expected']
GG_plot['fc'] = GG_plot['no_edges']/ GG_plot['expected']
GG_plot['log_p'] = -1*np.log10(GG_plot['p'])
GG_plot = GG_plot[(GG_plot.p!=0)&(GG_plot.no_edges_total<1000)]
GG_plot['log_fdr'] = -1*np.log10(fdr(GG_plot.p))
#GG_plot['log_fdr'] = -1*np.log10(GG_plot['fdr'])
GG_plot.loc[GG_plot[GG_plot.acc.isin(mdSAMs)].index,'mdSAM']  = '+'
GG_plot.loc[GG_plot[~GG_plot.acc.isin(mdSAMs)].index,'mdSAM']  = '-'
GG_plot['norm_obs'] = GG_plot['obs_exp']/GG_plot['no_edges_total']

In [ ]:
KeyWord_df = pd.read_csv('/Users/aelazar/Downloads/keywords_all_2023_10_27.tsv',delimiter='\t')
hTMs_uniprot = pd.read_csv('/Users/aelazar/Downloads/idmapping_2023_10_27.tsv',delimiter='\t')
hTMs_uniprot = hTMs_uniprot[~pd.isna(hTMs_uniprot.Keywords)]

In [ ]:
GG

In [ ]:
IDs = set(KeyWord_df[KeyWord_df.Category.isin(['Biological process','Molecular function','Disease'])]['Keyword ID'])
KEY = []
for j in hTMs_uniprot[(~pd.isna(hTMs_uniprot['Keyword ID']))].index: #(hTMs_uniprot.Entry.isin(mdSAM.acc))&
    #hTMs_uniprot.loc[0]['Gene Ontology (biological process)']
    t= [i.replace(' ','') for i in hTMs_uniprot.loc[j,'Keyword ID'].split(';')]
    if len(set.intersection(IDs,set(t)))>0:
        for key in set.intersection(IDs,set(t)):
            KEY.append({'ID':key,'disrcption':KeyWord_df[KeyWord_df['Keyword ID']==key].Name.values[0],'acc':hTMs_uniprot.loc[j,'Entry']})
KEYdf = pd.DataFrame(KEY)
KEYdf_counts = KEYdf.ID.value_counts().reset_index()

KEYdf_counts= KEYdf_counts.rename(columns={'ID': 'counts_total','index':'ID'})
for i in KEYdf_counts.index:
    id = KEYdf_counts.loc[i,'ID']
    KEYdf_counts.loc[i,'disrcption'] =KeyWord_df[KeyWord_df['Keyword ID']==id].Name.values[0]
top_IDs = KEYdf_counts[KEYdf_counts.counts_total.between(50,1500)]

In [ ]:

for ind in top_IDs.index:
    #print('%s\t%s'%(top_IDs.loc[ind,'disrcption'],top_IDs.loc[ind,'ID']))
    accs = hTMs_uniprot[hTMs_uniprot['Keyword ID'].str.find(top_IDs.loc[ind,'ID'])!=-1].Entry
    l = len(GG_plot[(GG_plot.obs_exp>0)&GG_plot.acc.isin(accs)])/len(GG_plot[GG_plot.acc.isin(accs)])
    if l<.35:
        print('%s\t%s'%(top_IDs.loc[ind,'disrcption'],top_IDs.loc[ind,'ID']))

In [ ]:
KeyWord_df#KW-0813



accs = hTMs_uniprot[hTMs_uniprot['Keyword ID'].str.find('KW-0829')!=-1].Entry

In [ ]:
KeyWord_df

In [ ]:
ps  = []
for ind in  top_IDs.index:
    accs = hTMs_uniprot[hTMs_uniprot['Keyword ID'].str.find(top_IDs.loc[ind,'ID'])!=-1].Entry
    #accs = hTMs_uniprot[hTMs_uniprot['Keyword ID'].str.find('KW-0829')!=-1].Entry
    #G_plot_PM_test =G_plot_PM[G_plot_PM.no_edges_total>1]
    GG_plot_sig = GG_plot[(GG_plot.log_fdr>.0)]
    if len(GG_plot_sig[GG_plot_sig.acc.isin(accs)])>20:
        a = len(GG_plot_sig[(GG_plot_sig.obs_exp>0)&(GG_plot_sig.acc.isin(accs))])
        b = len(GG_plot_sig[(GG_plot_sig.obs_exp<=0)&(GG_plot_sig.acc.isin(accs))])
        c = len(GG_plot_sig[(GG_plot_sig.obs_exp>0)&(~GG_plot_sig.acc.isin(accs))])
        d = len(GG_plot_sig[(GG_plot_sig.obs_exp<=0)&(~GG_plot_sig.acc.isin(accs))])
        table_key= [[a,b],[c,d]]
        
        expected_freq_key = expected_freq(table_key)
            
        #print(stats.fisher_exact(table_key,alternative='two-sided')[-1])
        #print(table_key,expected_freq_key)
        ps.append({'ID':top_IDs.loc[ind,'ID'],'p_val':stats.fisher_exact(table_key,alternative='two-sided')[-1],
                   'table':table_key,'expected_table':expected_freq(table_key),
                   'enrich':np.log2(table_key[0][0]/expected_freq_key[0][0])})

In [ ]:
ps_df = pd.DataFrame(ps)
ps_df['fdr'] = fdr(ps_df['p_val'])
ps_df['fdr_log'] = -1*np.log(ps_df['fdr'])
ps_df.sort_values(by='fdr_log')
ps_df['des'] = ps_df['ID'].map(KEYdf.set_index('ID').to_dict()['disrcption'])
#ps_df.sort_values(by='enrich').tail(10).loc[:,['des','enrich','fdr_log']]
fig, ax = plt.subplots(figsize=(3,3))
sns.scatterplot(x='enrich',y='fdr_log',data=ps_df,edgecolor=None)
plt.yscale('symlog')
plt.axhline(.6)
plt.axvline(.7)
plt.axvline(-.7)

In [ ]:
ps_df[(ps_df.enrich<-.6)]

In [ ]:
GG_plot[GG_plot.no_edges_total>0]
norm = Normalize(vmax=GG_plot[GG_plot.no_edges_total>min_number].loc[:,'no_edges_total'].max()
                 ,vmin=GG_plot[GG_plot.no_edges_total>min_number].loc[:,'no_edges_total'].min())#p
fig, axs = plt.subplot_mosaic("AB;AC",gridspec_kw = {'hspace':.2,'wspace':.1,'width_ratios': [2,1]},figsize = (10,6))
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=GG_plot[GG_plot.no_edges_total>min_number],rasterized=False,s=75,
                hue='no_edges_total',edgecolor=None,palette=ww,zorder=2,alpha=1,sizes='no_edges_total',ax=axs['A'])
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=GG_plot[GG_plot.acc.isin(hTMs_uniprot[hTMs_uniprot['Keyword ID'].str.find('KW-0813')!=-1].Entry)],rasterized=False
                ,edgecolor='k',c='k',s=20,zorder=2,ax=axs['B'],alpha=1)
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=GG_plot[~GG_plot.acc.isin(hTMs_uniprot[hTMs_uniprot['Keyword ID'].str.find('KW-0813')!=-1].Entry)],rasterized=False
                ,edgecolor=None,c='gray',s=20,zorder=1,ax=axs['B'])
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=GG_plot[GG_plot.acc.isin(hTMs_uniprot[hTMs_uniprot['Keyword ID'].str.find('KW-0418')!=-1].Entry)],rasterized=False
                ,edgecolor=None,c='#FDAF11',s=20,zorder=2,ax=axs['C'])
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=GG_plot[~GG_plot.acc.isin(hTMs_uniprot[hTMs_uniprot['Keyword ID'].str.find('KW-0418')!=-1].Entry)],rasterized=False
                ,edgecolor=None,c='gray',s=20,zorder=1,ax=axs['C'])
for i in ['A','B','C']:
    axs[i].grid(zorder=0)
    axs[i].set_yscale('log')
    axs[i].set_xlim([-30,30])
    axs[i].set_ylim([0,6])
    #axs[i].locator_params(nbins=6)
    if i!='A':
        axs[i].set_xticklabels([])
        axs[i].set_yticklabels([])
        axs[i].set_xlabel('')
        axs[i].set_ylabel('')
        axs[i].axhline(-1*np.log10(.25),ls='--',color='k',lw=2)
        axs[i].axvline(8,ls='--',color='k',lw=2)
        axs[i].axvline(-8,ls='--',color='k',lw=2)
axs['A'].set_yticklabels(axs['A'].get_yticklabels(),fontsize=18)
axs['A'].set_xticklabels(axs['A'].get_xticklabels(),fontsize=18)
axs['A'].set_ylabel('$‑log10(FDR)$',fontsize=20)
h,l = axs['A'].get_legend_handles_labels()
axs['A'].legend(h[:], l[:],title='Total No edges',frameon=True,fontsize=10
               )   # <<<<<<<< This is where the magic happens

axs['B'].set_title('Transport',fontsize=14)
axs['B'].axis('off')
axs['C'].axis('off')
axs['C'].set_title('Tyrosine-protein kinase',fontsize=14)
axs['A'].axhline(-1*np.log10(.25),ls='--',color='k',lw=2)
axs['A'].axvline(8,ls='--',color='k',lw=2)
axs['A'].axvline(-8,ls='--',color='k',lw=2)
#axs['A'].set_xlabel('$deg(Obs)-deg(Exp)$\n$Red\;edges$',fontsize=20)
axs['A'].set_xlabel('$Obs[\;\;\;]-Exp[\;\;\;]$',fontsize=20)

# axs['B'].text(10,1.7,'%d/%d'%(table_rho[0][0],expected_freq_rho[0][0]),fontsize=18)
# axs['C'].text(10,1.7,'%d/%d'%(table_endo_GO[0][0],expected_freq_endo[0][0]),fontsize=18)
texts = []
for i in GG_plot[(GG_plot.obs_exp>=8)&(GG_plot.log_fdr>-1*np.log10(.5))].index:
    texts.append(axs['A'].text(GG_plot.loc[i,'obs_exp'],GG_plot.loc[i,'log_fdr'],
            '%s'%GG_plot.loc[i,'gene'],
                          fontsize=12))
adjust_text(texts, autoalign='xy',ax=axs['A'],
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
texts = []
for i in GG_plot[(GG_plot.obs_exp<-8)&(GG_plot.log_fdr>-1*np.log10(.5))].index:
    texts.append(axs['A'].text(GG_plot.loc[i,'obs_exp'],GG_plot.loc[i,'log_fdr'],
            '%s'%GG_plot.loc[i,'gene'],
                          fontsize=12))
adjust_text(texts, autoalign='xy',ax=axs['A'],
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
#plt.savefig('Rho_enrich_PM_v7.svg',bbox_inches='tight',transparent=True,dpi=600)
# norm = Normalize(vmax=200,vmin=1)
# sm = plt.cm.ScalarMappable(cmap=ww, norm=norm)
# sm.set_array([])
# cbar = axs['A'].figure.colorbar(sm,shrink=.55)
# cbar.ax.tick_params(labelsize=12)
# cbar.set_label(label='Total No edges', size='large')
#plt.savefig('Rho_enrich_PM_v8.svg',bbox_inches='tight',transparent=True,dpi=600)
axs['A'].legend([],frameon=False)


# sns.scatterplot(x='obs_exp',y='log_fdr',
#                 data=GG_plot[GG_plot.no_edges_total>0],rasterized=False,s=75,
#                 hue='no_edges_total',edgecolor=None,palette=ww,zorder=2,alpha=1,sizes='no_edges_total')
# # plt.yscale('log')
# plt.axhline(-1*np.log10(.25))
# plt.axvline(-5)
# plt.axvline(5)
plt.savefig('Rho_enrich_PM_v8.svg',bbox_inches='tight',transparent=True,dpi=600)


In [ ]:
fig, ax = plt.subplots(figsize=(5,5)) 
norm = Normalize(vmax=GG_plot[GG_plot.no_edges_total>min_number].loc[:,'no_edges_total'].max()
                 ,vmin=GG_plot[GG_plot.no_edges_total>min_number].loc[:,'no_edges_total'].min())#p
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=GG_plot[GG_plot.no_edges_total>min_number],rasterized=False,s=75,
                hue='no_edges_total',edgecolor=None,palette=ww,zorder=2,alpha=1,sizes='no_edges_total',ax=ax)
plt.ylabel('-log$_{10}$(FDR)',fontsize=16)
#set_ylabel('$‑log10(FDR)$',fontsize=20)
plt.xlabel('',fontsize=16)
ax.legend([],frameon=False)
ax.grid()

#ax.set_ylim([0,10])
ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
#plt.yscale('log')
plt.axhline(-1*np.log10(.25),ls='--',c='k')
plt.axvline(-10,ls='--',c='k')
plt.axvline(10,c='k',ls='--')
norm = Normalize(vmax=192,vmin=1)
sm = plt.cm.ScalarMappable(cmap=ww, norm=norm)
sm.set_array([])
cbar = ax.figure.colorbar(sm,shrink=.45)
cbar.ax.tick_params(labelsize=12)
cbar.set_label(label='Total No edges', size='large')
texts = []
for i in GG_plot[(GG_plot.obs_exp>=10)&(GG_plot.log_fdr>-1*np.log10(.5))].index:
    texts.append(ax.text(GG_plot.loc[i,'obs_exp'],GG_plot.loc[i,'log_fdr'],
            '%s'%GG_plot.loc[i,'gene'],
                          fontsize=12))
adjust_text(texts, autoalign='xy',ax=ax,
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
texts = []
for i in GG_plot[(GG_plot.obs_exp<=-10)&(GG_plot.log_fdr>-1*np.log10(.5))].index:
    texts.append(ax.text(GG_plot.loc[i,'obs_exp'],GG_plot.loc[i,'log_fdr'],
            '%s'%GG_plot.loc[i,'gene'],
                          fontsize=12))
adjust_text(texts, autoalign='xy',ax=ax,
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
plt.savefig('PM_enrich_PM_v1.svg',bbox_inches='tight',transparent=True,dpi=600)

In [ ]:
fig, ax = plt.subplots(figsize=(4,4)) 
norm = Normalize(vmax=GG_plot[GG_plot.no_edges_total>min_number].loc[:,'no_edges_total'].max()
                 ,vmin=GG_plot[GG_plot.no_edges_total>min_number].loc[:,'no_edges_total'].min())#p
sns.scatterplot(x='obs_exp',y='log_fdr',
                data=GG_plot[GG_plot.no_edges_total>min_number],rasterized=False,s=75,
                edgecolor=None,color='gray',zorder=2,alpha=1,ax=ax)
sns.scatterplot(x='obs_exp',y='log_fdr',
                data= GG_plot[(GG_plot.obs_exp>=10)&(GG_plot.log_fdr>-1*np.log10(.5))],rasterized=False,s=75,
                edgecolor=None,color='#155289',zorder=2,alpha=1,ax=ax)
sns.scatterplot(x='obs_exp',y='log_fdr',
                data= GG_plot[(GG_plot.obs_exp<=-10)&(GG_plot.log_fdr>-1*np.log10(.5))],rasterized=False,s=75,
                edgecolor=None,color='#155289',zorder=2,alpha=1,ax=ax)
plt.ylabel('-log$_{10}$(FDR)',fontsize=16)
#set_ylabel('$‑log10(FDR)$',fontsize=20)
plt.xlabel('',fontsize=16)
ax.legend([],frameon=False)
ax.grid()

#ax.set_ylim([0,10])
ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
#plt.yscale('log')
plt.axhline(-1*np.log10(.25),ls='--',c='k')
plt.axvline(-10,ls='--',c='k')
plt.axvline(10,c='k',ls='--')
# norm = Normalize(vmax=192,vmin=1)
# sm = plt.cm.ScalarMappable(cmap=ww, norm=norm)
# sm.set_array([])
# cbar = ax.figure.colorbar(sm,shrink=.45)
# cbar.ax.tick_params(labelsize=12)
# cbar.set_label(label='Total No edges', size='large')
texts = []
for i in GG_plot[(GG_plot.obs_exp>=10)&(GG_plot.log_fdr>-1*np.log10(.5))].index:
    texts.append(ax.text(GG_plot.loc[i,'obs_exp'],GG_plot.loc[i,'log_fdr'],
            '%s'%GG_plot.loc[i,'gene'],
                          fontsize=12))
adjust_text(texts, autoalign='xy',ax=ax,
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
texts = []
for i in GG_plot[(GG_plot.obs_exp<=-10)&(GG_plot.log_fdr>-1*np.log10(.5))].index:
    texts.append(ax.text(GG_plot.loc[i,'obs_exp'],GG_plot.loc[i,'log_fdr'],
            '%s'%GG_plot.loc[i,'gene'],
                          fontsize=12))
adjust_text(texts, autoalign='xy',ax=ax,
            arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
plt.savefig('PM_enrich_PM_v1.svg',bbox_inches='tight',transparent=True,dpi=600)

In [ ]:
sns.kdeplot(hue='mdSAM',x='obs_exp',data=GG_plot,common_norm=False)

In [ ]:
set.intersection(mdSAMs_PM,set([i for i in GG['P51686']]))

In [ ]:
l = 0
for i in GG.nodes:
    l= l  + len([i for i in GG[i]])
l

In [ ]:
l = 0
for i in set.intersection(set(GG.nodes),mdSAM_PM):
    l= l  + len([i for i in GG[i]])
l

In [ ]:
total_runners =  29940 # No of Runners
number_team_A = 658# number runners in team A
number_winners = 603 # number of winners 
number_win_from_A = 111 # number of winners from team A 
#print(total_runners,number_team_A,number_winners,number_win_from_A)
expected = (number_team_A*number_winners)/total_runners
p = stats.hypergeom.cdf(number_win_from_A-1, total_runners, number_winners, number_team_A)
1-p

In [ ]:
GG_plot[GG_plot.no_edges_total>0].mdSAM.value_counts()

In [ ]:
len(GG_plot[(GG_plot.no_edges_total>min_number)&(GG_plot.mdSAM=='-')&(GG_plot.obs_exp<0)])/len(GG_plot[(GG_plot.no_edges_total>min_number)&(GG_plot.mdSAM=='-')])

In [ ]:
GG_plot.no_edges_total.sum() - GG_plot.no_edges.sum()#GG_plot.no_edges_total.sum()

In [ ]:
GG_plot.to_excel('GG_plot_mem_meb.xlsx')

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
sns.scatterplot(x='no_edges_total',y='obs_exp',data=GG_plot,hue='mdSAM',edgecolor=None,alpha=.81,s=60,palette=['#B4B5B4','#C03830'])
plt.xscale('log')
plt.ylabel('deg(Obs)-deg(Exp)',fontsize=16)
plt.xlabel('No. edges',fontsize=16)
plt.locator_params(axis='y', nbins=6)
ax.grid()
plt.axhline(0,zorder=2,color='k',ls='--')
plt.legend([],frameon=False)
plt.savefig('enrich_TM_PPI__PM_v4.svg',bbox_inches='tight',transparent=True)

In [ ]:
print('%d mdSAM nodes'%len(set.intersection(set(GG.nodes),mdSAM_PM_all)))
print('%d non-mdSAM nodes'%len(set.intersection(set(GG.nodes),set.difference(PM_all,mdSAM_PM_all))))
print('%d non-TM nodes'%len(set.intersection(set(GG.nodes),non_TMs_PM)))


In [ ]:
z = []
for i in range(1,60,1):
    GG3 = GG_plot[GG_plot.no_edges_total>=i]
    z.append({'min_no_edgds':i,'pre':100*(len(GG3[(GG3.norm_obs<0)&(GG3.mdSAM=='+')])/len(GG3[(GG3.mdSAM=='+')])),'ele':'+'})
    z.append({'min_no_edgds':i,'pre':100*(len(GG3[(GG3.norm_obs<0)&(GG3.mdSAM=='-')])/len(GG3[(GG3.mdSAM=='-')])),'ele':'-'})
fig, ax = plt.subplots(figsize=(5,5))
sns.lineplot(x='min_no_edgds',y='pre',data=pd.DataFrame(z),hue='ele',lw=4,palette=['#B4B5B4','#C03830'],hue_order=['-','+'],ax=ax)
#sns.scatterplot(x='min_no_edgds',y='pre',data=pd.DataFrame(z),hue='ele',palette=['#B4B5B4','#C03830'],hue_order=['-','+'],ax=ax)
plt.axhline(50,color='k',lw=3,ls='--')
ax.grid()
plt.ylabel('% of points below 0',fontsize=16)
plt.legend([],frameon=False)
plt.xlabel('Threshold(No.edges)',fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
plt.savefig('enrich_TM_PPI_below_zero_PM.svg',bbox_inches='tight',transparent=True)

In [ ]:
d3 = dd[dd.degree.between(1,200)]
fig, ax = plt.subplots(figsize=(3,6)) 
sns.boxplot(x='mdSAM',y='degree',data=d3
                  ,showfliers=False,zorder=2,
                 fliersize=0, width=0.9, medianprops=dict(color="k", alpha=1,zorder=2,linewidth=1), boxprops=dict(linewidth=1, facecolor='w', edgecolor='k', alpha=.61), 
                 whiskerprops=dict(linewidth=1, color='k', alpha=.6,zorder=3.75), capprops=dict(linewidth=1, color='k', alpha=.6,zorder=3),native_scale=True)
#sns.swarmplot(x='mdSAM',y='degree',data=d3,s=2,zorder=2,dodge=False,palette=['#B4B5B4','#C03830'],hue='mdSAM')
# plt.ylabel('Degree',fontsize=16)
# plt.xlabel('mdSAM',fontsize=16)
# ax.grid()
# ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
# ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
# plt.savefig('Dgree_MP-MP.svg',bbox_inches='tight',transparent=True,dpi=600)

In [ ]:
print('%d mdSAM nodes'%len(set.intersection(set(G_PM_All.nodes),mdSAM_PM_all)))
print('%d non-mdSAM nodes'%len(set.intersection(set(G_PM_All.nodes),set.difference(PM_all,mdSAM_PM_all))))
print('%d non-TM nodes'%len(set.intersection(set(GG.nodes),non_TMs_PM)))
print('%d non-TM edges'%len(set.intersection(set(GG.nodes),non_TMs_PM)))

In [ ]:
dd.degree.max()

In [ ]:
d3 = dd[dd.degree.between(1,200)]
stats.ks_2samp(d3[d3.mdSAM=='+'].degree,
              d3[d3.mdSAM=='-'].degree)

In [ ]:
stats.ks_2samp(d3[d3.mdSAM=='+'].degree,
              d3[d3.mdSAM=='-'].degree)


In [ ]:
MP_Sol_deg[MP_Sol_deg.mdSAM=='-']

In [ ]:
print(stats.ks_2samp(MP_Sol_deg[MP_Sol_deg.mdSAM=='+'].degree,
              MP_Sol_deg[MP_Sol_deg.mdSAM=='-'].degree))
print(stats.ks_2samp(d3[d3.mdSAM=='+'].degree,
              d3[d3.mdSAM=='-'].degree))

In [ ]:
fig = plt.figure(figsize = (8,4),constrained_layout=False)
axs = fig.subplots(nrows=1, ncols=2,gridspec_kw = {'hspace':.01,'wspace':.05})
sns.ecdfplot(hue='mdSAM',x='degree',data=MP_Sol_deg,
             log_scale=False,palette=['#B4B5B4','#C03830'],hue_order=['-','+'],lw=3,ax=axs[0])
sns.ecdfplot(hue='mdSAM',x='degree',data=d3,
             log_scale=False,palette=['#B4B5B4','#C03830'],hue_order=['-','+'],lw=3,ax=axs[1])
axs[0].set_xlim([-5,50])
axs[1].set_xlim([-5,50])
axs[1].set_yticklabels([])
axs[1].set_ylabel('')
axs[1].set_xlabel('')
axs[0].set_xlabel('Degree',fontsize=16)
axs[0].set_ylabel('Proportion',fontsize=16)
axs[1].set_xlabel('',fontsize=16)
axs[1].grid()
axs[0].grid()
#axs[0].legend(ncol=1,frameon=True)
axs[1].legend([],frameon=False)
axs[0].set_yticklabels(axs[0].get_yticklabels(),fontsize=16)
axs[1].set_xticklabels(axs[1].get_xticklabels(),fontsize=16)
axs[0].set_xticklabels(axs[0].get_xticklabels(),fontsize=16)
axs[0].set_title('Sol-Mem',fontsize=16)#Sol-Mem
axs[1].set_title('Mem-Mem',fontsize=16)#Mim-em
axs[1].text(20,.85,'*',fontsize=24)
axs[0].text(20,.85,'*',fontsize=24)
plt.savefig('Degree_KS2.svg',bbox_inches='tight',transparent=True)

In [ ]:
sns.violinplot(y='degree',data=MP_Sol_deg,hue='mdSAM',hue_order=[''])

In [ ]:
MP_Sol_deg['graph_type']='SOL-MEM'
d3['graph_type']='MEM-MEM'
dff_degree = pd.concat([MP_Sol_deg,d3])

In [ ]:
fig, ax = plt.subplots(figsize=(3,4))

sns.violinplot(x='graph_type',y='degree',hue='mdSAM',
               data=dff_degree,split=True,fill=0,hue_order=['-','+'],
              palette=['gray','r'])
plt.locator_params(nbins=6)
plt.ylim([-20,75])
plt.legend([])
ax.grid()
plt.savefig('Degree_KS2_v2.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig, ax = plt.subplots(figsize=(2,3))

sns.boxplot(x='graph_type',y='degree',hue='mdSAM',
               data=dff_degree,fill=0,hue_order=['-','+'],
              palette=['gray','r'])
sns.stripplot(x='graph_type',y='degree',hue='mdSAM',
               data=dff_degree,hue_order=['-','+'],
              palette=['gray','r'],dodge=True,jitter=False,s=2)
plt.locator_params(nbins=4)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlabel('',fontsize=12)
plt.ylabel('',fontsize=12)
plt.ylim([-10,220])
plt.legend([],frameon=False)
ax.grid()
plt.savefig('Degree_KS2_v2.svg',bbox_inches='tight',transparent=True)

In [ ]:
stats.ks_2samp(dff_degree[(dff_degree.graph_type=='MEM-MEM')&(dff_degree.mdSAM=='+')]['degree'],
dff_degree[(dff_degree.graph_type=='MEM-MEM')&(dff_degree.mdSAM=='-')]['degree'])

In [ ]:
print(stats.ks_2samp(MP_Sol_deg[MP_Sol_deg.mdSAM=='+'].degree,
              MP_Sol_deg[MP_Sol_deg.mdSAM=='-'].degree))
print(stats.ks_2samp(d3[d3.mdSAM=='+'].degree,
              d3[d3.mdSAM=='-'].degree))

In [ ]:
sns.ecdfplot(hue='mdSAM',x='degree',data=d3,log_scale=False,hue_order=['-','+'])
sns.ecdfplot(hue='mdSAM',x='degree',data=MP_Sol_deg,log_scale=False,hue_order=['-','+'])
plt.xlim([0,100])

In [ ]:
len(set(d3[d3.mdSAM=='+'].acc))

In [ ]:
G_plot_PM_all
G_plot_PM_all['norm_obs'] = G_plot_PM_all['obs_exp']/G_plot_PM_all['no_edges_total']

In [ ]:
GGG = nx.Graph()
l = []
#accs = (set.union(set(df.acc1),set(df.acc2))) ## Nodes 
for acc in set.intersection(set(df[(df.acc1.isin(sub_TMs))].acc1),
set(df[(df.acc2.isin(sub_TMs))].acc2)):#nodes_for_graph:
    if acc in list(mdSAMs):
        l.append((acc, {"mdSAM": "yes"}))
    else:
        l.append((acc, {"mdSAM": "no"}))
GGG.add_nodes_from(l)
for ind in df.index:
    if ((df.loc[ind,'acc1'] in sub_TMs) and (df.loc[ind,'acc2'] in sub_TMs)):
        if df.loc[ind,'acc1']==df.loc[ind,'acc2']:
            GGG.add_edge(df.loc[ind,'acc1'], df.loc[ind,'acc2'])
# z = []
# for i in GGG.nodes:
#     if len(GGG[i])==0:
#         z.append(i)
#         #G.remove_node(i)
# GGG.remove_nodes_from(z)

In [ ]:
self_inter = []
for i in GGG.edges:
    self_inter.append(i[0])

In [ ]:
len(set.intersection(set(df[(df.acc1.isin(sub_TMs))].acc1),
set(df[(df.acc2.isin(sub_TMs))].acc2)))

In [ ]:
def GGG_hypergeom(acc,mdSAMs):
    total_runners =  len(set(list(GGG.nodes))) # No. of hTMs Nodes in the Graph 
    number_team_A = len(set.intersection(set(list(GGG.nodes)),mdSAMs)) # number mdSAM nodes in the Graph  
    number_winners = len(list(GGG[acc]))## No. of PPIs i.e Edges 
    number_win_from_A = len(set.intersection(set(list(GGG[acc])),mdSAMs)) ## PPIs that are mdSAM
    #print(total_runners,number_team_A,number_winners,number_win_from_A)
    expected = (number_team_A*number_winners)/total_runners
    p = stats.hypergeom.cdf(number_win_from_A, total_runners, number_winners, number_team_A)
    #print(p)
    if  number_win_from_A-expected >0:
        p = 1-p 
    return number_win_from_A,expected,p 

In [ ]:
GGG_hypergeom_list =[]
for i in GGG.nodes:
    n,e,p  = GGG_hypergeom(i,mdSAMs)#pd.Series(sub_TMs).sample(969)#idr_df[(idr_df.idr.between(200,2000))&(~idr_df.acc.isin(mdSAMs))].acc
    if acc2name[acc2name.From==i]['Gene Names'].empty==False:
        gene = acc2name[acc2name.From==i]['Gene Names'].values[0].split(' ')[0]
    else:
        gene= np.NaN
    GGG_hypergeom_list.append({'acc':i,'no_edges':n,'expected':e,'p':p,'no_edges_total':len(set.intersection(set(list(GGG[i])),set(sub_TMs))),
                             'gene':gene,'round':'real'})

In [ ]:
GGG_plot = pd.DataFrame(GGG_hypergeom_list)
GGG_plot['obs_exp'] = (GGG_plot['no_edges']-GGG_plot['expected'])#/G_hypergeom_df['expected']
GGG_plot['fc'] = GGG_plot['no_edges']/ GGG_plot['expected']
GGG_plot['log_p'] = -1*np.log10(GGG_plot['p'])
#GGG_plot = GGG_plot[(GGG_plot.p!=0)&(GGG_plot.no_edges_total<300)]
from scipy.stats import fisher_exact
from scipy.stats.contingency import expected_freq
no_mdSAM_self = 348
mdSAM_self = 111
mdSAM = len(set.intersection(set(list(GGG.nodes)),mdSAMs))
no_mdSAM = len(set.difference(set(list(GGG.nodes)),mdSAMs))


table = [[mdSAM_self, mdSAM-mdSAM_self], [no_mdSAM_self, no_mdSAM-no_mdSAM_self]]
res = fisher_exact(table)


print(res[1])
expected_table = expected_freq(table)
table-expected_table

sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})

fig, ax = plt.subplots(figsize=(3,5)) 
bar = sns.barplot(x='x',y='val',hue='ele',
            data=pd.DataFrame([{'x':'box','val':table[0][0],'ele':'obs'},{'x':'box','val':expected_table[0][0],'ele':'exp'}])
           ,hue_order=['obs','exp'],lw=0,palette=['#C03830','k'])
plt.ylabel('$No.\;edges$')
plt.xlabel('')
ax.grid()
ax.set_xticklabels([''])
# e= 'p=%.E'%res[1]

hatches = ['\\\\' for i in range(2)]

# Loop over the bars
c = 0
colors = ['#C03830','C03830','#C03830']
for i,thisbar in enumerate(bar.patches):
    # Set a different hatch for each bar
    if i in [1]:
        
        thisbar.set_facecolor(colors[c])
        thisbar.set_hatch(hatches[i])
        thisbar.set_edgecolor('white')
        c = c+1
c = 0
for i,thisbar in enumerate(bar.patches):
    # Set a different hatch for each bar
    if i not in [1]:
        thisbar.set_facecolor(colors[c])
        c = c+1
plt.rcParams['hatch.linewidth'] = 1.5
L=plt.legend(fontsize=12)
L.get_texts()[0].set_text('Obs')
L.get_texts()[1].set_text('Exp')

ax.set_yticklabels(ax.get_yticklabels(),fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=16)
# e = '%1.10e' %res[1]
# lead = e[0]
# power = e.split('-')[-1].split('0')[-1]
# #plt.text(i[0]-.3,5,'$%s⋅10^{‑%s}$'%(lead,power),rotation=90,fontsize=10,color='k')
# plt.text(0.1,61,'$p=%s⋅10^{‑%s}$'%(lead,power),rotation=90,fontsize=10,color='w')
# plt.title('$No.(Obs)\;vs\;No.(Exp)$\n$Red\;edges$',fontsize=10)
#ax.set_ylim([50,120])
#plt.locator_params(axis='y', nbins=6)
plt.savefig('self_association_BioGrid_v3.svg',bbox_inches='tight',transparent=True)

In [ ]:
GGG_plot

In [ ]:
fig, ax = plt.subplots(figsize=(3,5)) 
bar = sns.barplot(x='x',y='val',hue='ele',
            data=pd.DataFrame([{'x':'box','val':table[0][0],'ele':'obs'},{'x':'box','val':expected_table[0][0],'ele':'exp'}])
           ,hue_order=['obs','exp'],lw=0,palette=['#C03830','k'])

In [ ]:
len(set.difference(set(list(GGG.nodes)),mdSAMs))

In [ ]:
len(set.difference(set(self_inter),set(mdSAMs)))

In [ ]:
len(set.difference(set(self_inter),set(mdSAMs)))

In [ ]:
no_mdSAM_self = 348
mdSAM_self = 111
mdSAM = 658 #len(set.intersection(set(list(GGG.nodes)),mdSAMs))
no_mdSAM = 2642#len(set.difference(set(list(GGG.nodes)),mdSAMs))



table = [[mdSAM_self, mdSAM-mdSAM_self], [no_mdSAM_self, no_mdSAM-no_mdSAM_self]]
res = fisher_exact(table)
expected_freq(table)

In [ ]:
res

In [ ]:
# Replace these with your actual computed values
data = {
    'Edge Type': ['yes', 'no'],
    'exp': [expected_freq(table)[0][0], expected_freq(table)[1][0],
    'obs': [table[0][0], table[1][0],
    'odds_ratio':[oddsratio_yy,oddsratio_nn,oddsratio_yn]
}
plot_df_odds = pd.DataFrame(data)
#plot_df_odds.set_index('Edge Type', inplace=True)
plot_df_odds
fig, ax = plt.subplots(figsize=(3,3)) 
sns.barplot(y='Edge Type',x='odds_ratio',data=plot_df_odds,order=['no-no','yes-no','yes-yes'])
plt.axvline(1)
plt.locator_params(axis='x',nbins=4)
plt.savefig('Membrane_graph_BioGrid_v6.svg',bbox_inches='tight',transparent=True)

In [ ]:
plot_df_odds

In [ ]:
expected_table = expected_freq(table)
table-expected_table

In [ ]:
l = []
l.append({'val':111,'ele':'obs','ele';})
l.append({'val':91.52181818,'ele':'exp'})

In [ ]:
fig, ax = plt.subplots(figsize=(2,3))
sns.barplot(y='val',hue='ele',data=pd.DataFrame(l),hue_order=['obs','exp'],palette=['r','k'])
#plt.savefig('self_association_BioGrid_v4.svg',bbox_inches='tight',transparent=True)
plt.setp(ax.patches, linewidth=0)
plt.locator_params(axis='y', nbins=4)
ax.set_xticklabels(ax.get_xticklabels(),fontsize=12)
ax.set_yticklabels(ax.get_yticklabels(),fontsize=12)
plt.savefig('self_association_BioGrid_v4.svg',bbox_inches='tight',transparent=True)

In [ ]:
len(mdSAMs)

In [ ]:
def Graph_organelle_E3(organelle=PM,mdSAMs=mdSAMs,min_egde = 10, max_edge=90):
    sub_TMs = list(organelle)
    nodes_for_graph = set.union(set(E3_ligases),set(sub_TMs))
    mdSAM_PM  = set.intersection(organelle,mdSAMs)#(set.intersection(set(HPA_frac_df[HPA_frac_df.gmm_new==0].acc),set(dSAMs.acc)))
    non_mdSAM_PM  = set.difference(set(sub_TMs),mdSAM_PM)

    G = nx.Graph()
    l = []
    #accs = (set.union(set(df.acc1),set(df.acc2))) ## Nodes 
    for acc in nodes_for_graph:
        if acc in list(mdSAM_PM):
            l.append((acc, {"mdSAM": "yes"}))
        if acc in list(non_mdSAM_PM):
            l.append((acc, {"mdSAM": "no"}))
    G.add_nodes_from(l)
    for ind in tqdm.tqdm(df.index):
        if ((df.loc[ind,'acc1'] in nodes_for_graph) and (df.loc[ind,'acc2'] in nodes_for_graph)):
            if ((df.loc[ind,'acc1'] in sub_TMs) and (df.loc[ind,'acc2'] in sub_TMs)):
                continue#print(df.loc[ind,'acc1'],df.loc[ind,'acc2'])
            if ((df.loc[ind,'acc1']  not in sub_TMs) and (df.loc[ind,'acc2'] not in sub_TMs)):
                continue#print(df.loc[ind,'acc1'],df.loc[ind,'acc2'])
            else:
                G.add_edge(df.loc[ind,'acc1'], df.loc[ind,'acc2'])
    z = []
    for i in G.nodes:
        if len(G[i])<=min_egde:#en(set.intersection(set(sub_TMs),set([i for i in G[i]])))
            z.append(i)
        if len(G[i])>=max_edge:
            z.append(i)
            #G.remove_node(i)
    G.remove_nodes_from(z)
    # zz=[]
    # for i in G.nodes:
    #     if len(G[i])<=1:
    #         zz.append(i)
    # G.remove_nodes_from(zz)
    return G
def G2E3df(G,mdSAMs = mdSAMs):
    G_hypergeom_list = []
    for i in tqdm.tqdm(set.intersection(set(list(G.nodes)),set(E3_ligases))):#set.intersection(set(E3_ligases),set(non_TMs_PM))
        n,e,p  = G_hypergeom(G,i,mdSAMs)
        if acc2name[acc2name.From==i]['Gene Names'].empty==False:
            gene = acc2name[acc2name.From==i]['Gene Names'].values[0].split(' ')[0]
        else:
            gene= np.NaN
        G_hypergeom_list.append({'acc':i,'no_edges':n,'expected':e,'p':p,'no_edges_total':len(set.intersection(set(list(G[i])),set(sub_TMs))),
                                 'gene':gene,'round':'real'})
    G_hypergeom_df  = pd.DataFrame(G_hypergeom_list)
    G_hypergeom_df = G_hypergeom_df[G_hypergeom_df.no_edges_total>=3]
    G_hypergeom_df['obs_exp'] = (G_hypergeom_df['no_edges']-G_hypergeom_df['expected'])#/G_hypergeom_df['expected']
    G_hypergeom_df['fc'] = G_hypergeom_df['no_edges']/ G_hypergeom_df['expected']
    G_hypergeom_df['log_p'] = -1*np.log10(G_hypergeom_df['p'])
    G_hypergeom_df['fdr'] = fdr(G_hypergeom_df.p)
    G_hypergeom_df['log_fdr'] = -1*np.log10(G_hypergeom_df['fdr'])
    
    G_plot = G_hypergeom_df#G_hypergeom_df[(G_hypergeom_df.p!=0)&(G_hypergeom_df.no_edges_total<320)]
    G_plot  = G_plot[~pd.isna(G_plot.gene)]
    return G_plot

In [ ]:
G_PM_E3 = Graph_organelle_E3(organelle=PM_TMs,min_egde=1,max_edge=200)#10,200
mdSAM_PM_E3  = set.intersection(set(G_PM_E3.nodes),mdSAMs)
G_plot_PM_E3 = G2E3df(G_PM_E3,mdSAMs=mdSAM_PM_E3)

In [ ]:
len(G_PM_E3.nodes)

In [ ]:
total_runners =  625 # No. of hTMs Nodes in the Graph 
number_team_A = 160# number mdSAM nodes in the Graph  
number_winners = 511# No. of PPIs i.e Edges 
number_win_from_A = 49 ## PPIs that are mdSAM
#print(total_runners,number_team_A,number_winners,number_win_from_A)
expected = (number_team_A*number_winners)/total_runners
p = stats.hypergeom.cdf(number_win_from_A-1, total_runners, number_winners, number_team_A)
expected/number_win_from_A

In [ ]:
# Plotting the hypergeometric distribution
x = np.arange(0, number_team_A + 1)
pmf = stats.hypergeom.pmf(x, total_runners, number_winners, number_team_A)

plt.figure(figsize=(3,3))
plt.plot(x, pmf, 'k-', label='Hypergeometric PMF')
plt.axvline(expected, color='orange', linestyle='--', label=f'Expected ({expected:.1f})')
plt.axvline(number_win_from_A, color='#985042', linestyle='-', label=f'Observed ({number_win_from_A})')
# plt.title('Hypergeometric Distribution')
# plt.xlabel('Number of mdSAM PPIs (Wins)')
# plt.ylabel('Probability')
plt.legend()
#plt.xlim([50,130])
plt.locator_params(axis='y',nbins=4)
#plt.grid(True)
#plt.savefig('self_association_BioGrid_v5.svg',bbox_inches='tight',transparent=True)

In [ ]:
l = 0
for i in set.intersection(set(G_PM_E3),set(mdSAM_PM_E3)):
    
    if len(set.intersection(mdSAM_PM_E3,set([i  for i in G_PM_E3[i]])))>0:
        l = l+1 
l

In [ ]:
set.inset(PM_TMs),set(G_PM_E3.nodes)

In [ ]:
l = 0
for i in set.difference(set(G_PM_E3),set(mdSAM_PM_E3)):
    if len(set.intersection(set(E3_ligases),set([i  for i in G_PM_E3[i]])))>0:
        l = l +1
l

In [ ]:
sns.scatterplot(x='obs_exp',y='log_fdr',data=G_plot_PM_E3)

In [ ]:
G_plot_PM_E3.sort_values(by='obs_exp',ascending=False)

In [ ]:
total_runners =  3300 # No. of hTMs Nodes in the Graph 
number_team_A = 658# number mdSAM nodes in the Graph  
number_winners = 459# No. of PPIs i.e Edges 
number_win_from_A = 111 ## PPIs that are mdSAM
#print(total_runners,number_team_A,number_winners,number_win_from_A)
expected = (number_team_A*number_winners)/total_runners
p = stats.hypergeom.cdf(number_win_from_A-1, total_runners, number_winners, number_team_A)
1-p
# Plotting the hypergeometric distribution
x = np.arange(0, number_team_A + 1)
pmf = stats.hypergeom.pmf(x, total_runners, number_winners, number_team_A)

plt.figure(figsize=(3,3))
plt.plot(x, pmf, 'k-', label='Hypergeometric PMF')
plt.axvline(expected, color='orange', linestyle='--', label=f'Expected ({expected:.1f})')
plt.axvline(number_win_from_A, color='#985042', linestyle='-', label=f'Observed ({number_win_from_A})')
plt.title('Hypergeometric Distribution')
plt.xlabel('Number of mdSAM PPIs (Wins)')
plt.ylabel('Probability')
plt.legend()
plt.xlim([50,130])
plt.locator_params(axis='y',nbins=4)
#plt.grid(True)
plt.savefig('self_association_BioGrid_v5.svg',bbox_inches='tight',transparent=True)

In [ ]:
total_runners =  3300 # No. of hTMs Nodes in the Graph 
number_team_A = 2642# number mdSAM nodes in the Graph  
number_winners = 459# No. of PPIs i.e Edges 
number_win_from_A = 348 ## PPIs that are mdSAM
#print(total_runners,number_team_A,number_winners,number_win_from_A)
expected = (number_team_A*number_winners)/total_runners
p = stats.hypergeom.cdf(number_win_from_A-1, total_runners, number_winners, number_team_A)
1-p
# Plotting the hypergeometric distribution
x = np.arange(0, number_team_A + 1)
pmf = stats.hypergeom.pmf(x, total_runners, number_winners, number_team_A)

plt.figure(figsize=(3,3))
plt.plot(x, pmf, 'k-', label='Hypergeometric PMF')
plt.axvline(expected, color='orange', linestyle='--', label=f'Expected ({expected:.1f})')
plt.axvline(number_win_from_A, color='#B2B1B2', linestyle='-', label=f'Observed ({number_win_from_A})')
plt.title('Hypergeometric Distribution')
plt.xlabel('Number of mdSAM PPIs (Wins)')
plt.ylabel('Probability')
plt.legend()
plt.xlim([320,400])
plt.locator_params(axis='y',nbins=4)
#plt.grid(True)
plt.savefig('nomdSAM_self_association_BioGrid_v5.svg',bbox_inches='tight',transparent=True)

In [ ]:
from scipy.stats import fisher_exact

# Observed values
a = 111         # mdSAM with self-loop
b = 547         # mdSAM without self-loop
c = 348         # non-mdSAM with self-loop
d = 2294        # non-mdSAM without self-loop

# Contingency table
table_p = [[a, b],
         [c, d]]

# Fisher's Exact Test (test for enrichment in mdSAM)
oddsratio_dsam, p_value = fisher_exact(table_p, alternative='greater')

print("Fisher's Exact Test (Self-loop enrichment in mdSAM):")
print(f"p = {p_value:.4e}, odds ratio = {oddsratio:.2f}")


In [ ]:
oddsratio

In [ ]:
from scipy.stats import fisher_exact

# Original values
a = 111    # mdSAM with self-loop
b = 547    # mdSAM without self-loop
c = 348    # non-mdSAM with self-loop
d = 2294   # non-mdSAM without self-loop

# Flip the table: focus on non-mdSAM
table = [[c, d],  # non-mdSAM
         [a, b]]  # mdSAM

# Fisher's Exact Test: are non-mdSAMs enriched in self-loops?
oddsratio_non, p_value = fisher_exact(table, alternative='less')

print("Fisher's Exact Test (Self-loop enrichment in non-mdSAM):")
print(f"p = {p_value:.4e}, odds ratio = {oddsratio:.2f}")


In [ ]:
# Replace these with your actual computed values
data = {
    'Edge Type': ['yes-yes', 'no-no'],
    'exp': [expected_freq(table_p)[0][0], expected_freq(table)[0][0]],
    'obs': [table_p[0][0], table[0][0]],
    'odds_ratio':[oddsratio_dsam,oddsratio_non]
}
plot_df_odds = pd.DataFrame(data)
#plot_df_odds.set_index('Edge Type', inplace=True)
plot_df_odds
fig, ax = plt.subplots(figsize=(3,3)) 
sns.barplot(y='Edge Type',x='odds_ratio',data=plot_df_odds,order=['no-no','yes-yes'])
plt.axvline(1)
plt.locator_params(axis='x',nbins=4)
plt.savefig('Membrane_graph_self_BioGrid_v6.svg',bbox_inches='tight',transparent=True)

In [ ]:
plot_df_odds